In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, json, math, re, time, sys, subprocess, pkgutil, traceback, html
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

if pkgutil.find_loader("gradio") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44.0"])
import gradio as gr

/tmp/ipykernel_47320/667445572.py:12: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader("gradio") is None:


In [3]:
import os
import glob
import zipfile

def find_exported_model_dir():
    candidate_dirs = [
        "/content/B7_light_best_model",
        "/content/drive/MyDrive/B7_light_best_model",
    ]

    for d in candidate_dirs:
        if os.path.exists(os.path.join(d, "export_meta.json")) and os.path.exists(os.path.join(d, "custom_heads.pt")):
            return d

    search_roots = ["/content", "/content/drive/MyDrive"]
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for meta_path in glob.glob(os.path.join(root, "**", "export_meta.json"), recursive=True):
            model_dir = os.path.dirname(meta_path)
            if os.path.exists(os.path.join(model_dir, "custom_heads.pt")):
                return model_dir

    zip_candidates = [
        "/content/B7_light_best_model.zip",
        "/content/drive/MyDrive/B7_light_best_model.zip",
    ]
    for zip_path in zip_candidates:
        if os.path.exists(zip_path):
            extract_dir = os.path.splitext(zip_path)[0]
            if not os.path.exists(os.path.join(extract_dir, "export_meta.json")):
                with zipfile.ZipFile(zip_path, "r") as zf:
                    zf.extractall(extract_dir)
                inner_meta = glob.glob(os.path.join(extract_dir, "**", "export_meta.json"), recursive=True)
                if inner_meta:
                    return os.path.dirname(inner_meta[0])
            else:
                return extract_dir

    raise FileNotFoundError(
        "Không tìm thấy thư mục model export từ notebook train. "
        "Cần có export_meta.json và custom_heads.pt."
    )

MODEL_DIR = find_exported_model_dir()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using MODEL_DIR =", MODEL_DIR)


Using MODEL_DIR = /content/drive/MyDrive/B7_light_best_model


In [4]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    META = json.load(f)

MODEL_NAME = META["model_name"]
MAX_LENGTH = META.get("max_length", 1536)
HEAD_DROPOUT = META.get("head_dropout", 0.1)

TARGET_COLS = META.get("target_cols", ["TR", "CC", "LR", "GRA"])

TR_FEATURE_COLS = META["tr_feature_cols"]
CC_FEATURE_COLS = META["cc_feature_cols"]
LR_FEATURE_COLS = META["lr_feature_cols"]
GRA_FEATURE_COLS = META["gra_feature_cols"]

tr_feat_mean = np.array(META["tr_feat_mean"], dtype=np.float32)
tr_feat_std  = np.array(META["tr_feat_std"], dtype=np.float32)

cc_feat_mean = np.array(META["cc_feat_mean"], dtype=np.float32)
cc_feat_std  = np.array(META["cc_feat_std"], dtype=np.float32)

lr_feat_mean = np.array(META["lr_feat_mean"], dtype=np.float32)
lr_feat_std  = np.array(META["lr_feat_std"], dtype=np.float32)

gra_feat_mean = np.array(META["gra_feat_mean"], dtype=np.float32)
gra_feat_std  = np.array(META["gra_feat_std"], dtype=np.float32)

SCORE_MIN = 4.0
SCORE_MAX = 9.0
POOLING = META.get("pooling", "last_token")
FUSION_TYPE = META.get("fusion_type", "gated_score_fusion")

print("MODEL_NAME =", MODEL_NAME)
print("TARGET_COLS =", TARGET_COLS)
print("POOLING =", POOLING)
print("FUSION_TYPE =", FUSION_TYPE)


MODEL_NAME = Qwen/Qwen2.5-3B-Instruct
TARGET_COLS = ['TR', 'CC', 'LR', 'GRA']
POOLING = last_token
FUSION_TYPE = gated_score_fusion


In [5]:
STOPWORDS = {
    "a","an","the","and","or","but","if","while","is","am","are","was","were",
    "be","been","being","of","to","in","on","for","with","as","at","by","from",
    "that","this","these","those","it","its","he","she","they","them","their",
    "we","our","you","your","i","me","my","mine","his","her","hers","do","does",
    "did","have","has","had","will","would","can","could","should","may","might",
    "not","so","than","then","there","here","about","into","over","after","before",
    "more","most","some","any","such","no","nor","too","very"
}

DISCOURSE_MARKERS = [
    "however", "therefore", "moreover", "furthermore", "in addition",
    "for example", "for instance", "on the one hand", "on the other hand",
    "in conclusion", "to conclude", "in summary", "as a result",
    "firstly", "secondly", "finally", "besides", "nevertheless",
    "thus", "overall", "in contrast", "for this reason"
]

LONG_WORD_MIN_LEN = 7

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def normalize_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_words(text):
    text = normalize_text(text)
    return re.findall(r"[a-zA-Z']+", text)

def split_sentences(text):
    text = safe_text(text).strip()
    if not text:
        return []
    sents = re.split(r'(?<=[.!?])\s+', text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents

def split_paragraphs(text):
    text = safe_text(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) == 0 and text.strip():
        paras = [text.strip()]
    return paras

def word_count(text):
    return len(tokenize_words(text))

def sentence_count(text):
    return len(split_sentences(text))

def unique_ratio(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)

def root_ttr(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / math.sqrt(len(words))

def repetition_ratio(words):
    if len(words) <= 1:
        return 0.0
    c = Counter(words)
    repeated = sum(v for v in c.values() if v > 1)
    return repeated / len(words)

def repeated_word_ratio(words):
    if len(words) <= 1:
        return 0.0
    repeated_pairs = 0
    for i in range(1, len(words)):
        if words[i] == words[i - 1]:
            repeated_pairs += 1
    return repeated_pairs / len(words)

def avg_word_len(words):
    if len(words) == 0:
        return 0.0
    return float(np.mean([len(w) for w in words]))

def lexical_density_proxy(words):
    if len(words) == 0:
        return 0.0
    content_like = [w for w in words if len(w) > 3 and w not in STOPWORDS]
    return len(content_like) / len(words)

def long_word_ratio(words, min_len=LONG_WORD_MIN_LEN):
    if len(words) == 0:
        return 0.0
    return sum(len(w) >= min_len for w in words) / len(words)

def sentence_lengths(sentences):
    return [len(tokenize_words(s)) for s in sentences if len(tokenize_words(s)) > 0]

def count_discourse_markers(text):
    low = normalize_text(text)
    counts = []
    found = 0
    found_types = 0
    for m in DISCOURSE_MARKERS:
        c = low.count(m)
        counts.append(c)
        if c > 0:
            found += c
            found_types += 1
    return found, found_types

def prompt_keywords(prompt):
    words = tokenize_words(prompt)
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return set(words)

def jaccard_coverage(prompt, essay):
    pk = prompt_keywords(prompt)
    ew = set(tokenize_words(essay))
    if len(pk) == 0:
        return 0.0
    return len(pk & ew) / len(pk)

def prompt_essay_similarity(prompt, essay):
    pw = prompt_keywords(prompt)
    ew = set([w for w in tokenize_words(essay) if w not in STOPWORDS and len(w) > 2])
    if len(pw) == 0 or len(ew) == 0:
        return 0.0
    return len(pw & ew) / math.sqrt(len(pw) * len(ew))

def has_opinion_statement(text):
    low = normalize_text(text)
    patterns = [
        "i believe", "i think", "in my opinion", "from my perspective",
        "personally", "it seems to me", "i would argue"
    ]
    return any(p in low for p in patterns)

def has_both_views(text):
    low = normalize_text(text)
    return (
        ("on the one hand" in low and "on the other hand" in low) or
        ("while some people think" in low and "others believe" in low) or
        ("some people argue" in low and "others believe" in low)
    )

def has_example(text):
    low = normalize_text(text)
    return any(p in low for p in ["for example", "for instance", "such as", "to illustrate"])

def has_conclusion(text):
    low = normalize_text(text)
    return any(p in low for p in ["in conclusion", "to conclude", "to sum up", "overall"])

def punct_density(text):
    words = tokenize_words(text)
    if len(words) == 0:
        return 0.0
    puncts = re.findall(r"[.,;:!?]", safe_text(text))
    return len(puncts) / len(words)

def repeated_punct_ratio(text):
    raw = safe_text(text)
    if len(raw) == 0:
        return 0.0
    repeated = re.findall(r"([!?.,;:])\1+", raw)
    return len(repeated) / len(raw)

def lowercase_sentence_start_ratio(text):
    sents = split_sentences(text)
    if len(sents) == 0:
        return 0.0
    bad = 0
    checked = 0
    for s in sents:
        m = re.search(r"[A-Za-z]", s)
        if m:
            checked += 1
            ch = s[m.start()]
            if ch.islower():
                bad += 1
    if checked == 0:
        return 0.0
    return bad / checked

def lowercase_i_ratio(text):
    toks = re.findall(r"\b[a-zA-Z']+\b", safe_text(text))
    if len(toks) == 0:
        return 0.0
    bad = sum(1 for t in toks if t == "i")
    return bad / len(toks)

def missing_terminal_punct(text):
    txt = safe_text(text).strip()
    if not txt:
        return 1.0
    return 0.0 if txt[-1] in ".!?" else 1.0

def extract_tr_features(prompt, essay):
    return {
        "tr_prompt_essay_sim": float(prompt_essay_similarity(prompt, essay)),
        "tr_prompt_keyword_coverage": float(jaccard_coverage(prompt, essay)),
        "tr_has_opinion": float(has_opinion_statement(essay)),
        "tr_has_both_views": float(has_both_views(essay)),
        "tr_has_example": float(has_example(essay)),
        "tr_has_conclusion": float(has_conclusion(essay)),
        "tr_word_count": float(word_count(essay)),
    }

def extract_cc_features(prompt, essay):
    paras = split_paragraphs(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)
    dm_count, dm_div = count_discourse_markers(essay)

    para_lens = [word_count(p) for p in paras] if len(paras) > 0 else [0]

    return {
        "cc_num_paragraphs": float(len(paras)),
        "cc_avg_paragraph_len": float(np.mean(para_lens)) if len(para_lens) > 0 else 0.0,
        "cc_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_sentence_len_std": float(np.std(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_discourse_marker_count": float(dm_count),
        "cc_discourse_marker_diversity": float(dm_div),
    }

def extract_lr_features(prompt, essay):
    words = tokenize_words(essay)

    return {
        "lr_root_ttr": float(root_ttr(words)),
        "lr_avg_word_len": float(avg_word_len(words)),
        "lr_long_word_ratio": float(long_word_ratio(words)),
        "lr_repetition_ratio": float(repetition_ratio(words)),
        "lr_unique_word_ratio": float(unique_ratio(words)),
        "lr_lexical_density_proxy": float(lexical_density_proxy(words)),
    }

def extract_gra_features(prompt, essay):
    words = tokenize_words(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)

    return {
        "gf_word_count": float(len(words)),
        "gf_sentence_count": float(len(sents)),
        "gf_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_short_sentence_ratio": float(sum(l < 8 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_long_sentence_ratio": float(sum(l > 30 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_punct_density": float(punct_density(essay)),
        "gf_repeated_punct_ratio": float(repeated_punct_ratio(essay)),
        "gf_lowercase_sent_start_ratio": float(lowercase_sentence_start_ratio(essay)),
        "gf_lowercase_i_ratio": float(lowercase_i_ratio(essay)),
        "gf_repeated_word_ratio": float(repeated_word_ratio(words)),
        "gf_missing_terminal_punct": float(missing_terminal_punct(essay)),
    }

def build_input_text(prompt, essay):
    prompt = str(prompt).strip()
    essay = str(essay).strip()
    return (
        "You are an IELTS Writing examiner. "
        "Assess the essay based on four criteria: "
        "Task Response (TR), Coherence and Cohesion (CC), "
        "Lexical Resource (LR), and Grammatical Range and Accuracy (GRA).\n\n"
        "[PROMPT]\n"
        f"{prompt}\n\n"
        "[ESSAY]\n"
        f"{essay}"
    )

def standardize_features(feat_dict, cols, mean_, std_):
    arr = np.array([feat_dict[c] for c in cols], dtype=np.float32)
    std_ = np.where(std_ < 1e-6, 1.0, std_)
    arr = (arr - mean_) / std_
    return arr

def round_to_half(x):
    return round(float(x) * 2) / 2

In [6]:
class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        base_model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = False

        self.backbone = PeftModel.from_pretrained(base_model, MODEL_DIR)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        self.tr_feat_dim = len(TR_FEATURE_COLS)
        self.cc_feat_dim = len(CC_FEATURE_COLS)
        self.lr_feat_dim = len(LR_FEATURE_COLS)
        self.gra_feat_dim = len(GRA_FEATURE_COLS)

        # ===== TR =====
        self.tr_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.tr_feat_head = nn.Sequential(
            nn.Linear(self.tr_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.tr_gate = nn.Sequential(
            nn.Linear(hidden_size + self.tr_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        # ===== CC =====
        self.cc_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.cc_feat_head = nn.Sequential(
            nn.Linear(self.cc_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.cc_gate = nn.Sequential(
            nn.Linear(hidden_size + self.cc_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        # ===== LR =====
        self.lr_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.lr_feat_head = nn.Sequential(
            nn.Linear(self.lr_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.lr_gate = nn.Sequential(
            nn.Linear(hidden_size + self.lr_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        # ===== GRA =====
        self.gra_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.gra_feat_head = nn.Sequential(
            nn.Linear(self.gra_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.gra_gate = nn.Sequential(
            nn.Linear(hidden_size + self.gra_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def _fuse_score(self, pooled, feat, llm_head, feat_head, gate_net):
        llm_score = llm_head(pooled)                     # [B, 1]
        feat_score = feat_head(feat)                    # [B, 1]

        gate_input = torch.cat([pooled, feat], dim=1)   # [B, H + F]
        gate = torch.sigmoid(gate_net(gate_input))      # [B, 1]

        fused_score = gate * llm_score + (1.0 - gate) * feat_score
        return fused_score, gate

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        tr_features=None,
        cc_features=None,
        lr_features=None,
        gra_features=None,
        labels=None,
        **kwargs
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        head_dtype = self.tr_llm_head[0].weight.dtype
        pooled = pooled.to(dtype=head_dtype)

        tr_features = tr_features.to(device=pooled.device, dtype=head_dtype)
        cc_features = cc_features.to(device=pooled.device, dtype=head_dtype)
        lr_features = lr_features.to(device=pooled.device, dtype=head_dtype)
        gra_features = gra_features.to(device=pooled.device, dtype=head_dtype)

        tr, tr_gate = self._fuse_score(
            pooled, tr_features, self.tr_llm_head, self.tr_feat_head, self.tr_gate
        )
        cc, cc_gate = self._fuse_score(
            pooled, cc_features, self.cc_llm_head, self.cc_feat_head, self.cc_gate
        )
        lr, lr_gate = self._fuse_score(
            pooled, lr_features, self.lr_llm_head, self.lr_feat_head, self.lr_gate
        )
        gra, gra_gate = self._fuse_score(
            pooled, gra_features, self.gra_llm_head, self.gra_feat_head, self.gra_gate
        )

        combined_logits = torch.cat([tr, cc, lr, gra], dim=1)

        return {
            "logits": combined_logits,
            "reg_logits": combined_logits,
            "tr_gate": tr_gate,
            "cc_gate": cc_gate,
            "lr_gate": lr_gate,
            "gra_gate": gra_gate,
        }


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = QwenForIELTSMultiTask(
    MODEL_NAME,
    tokenizer,
    head_dropout=HEAD_DROPOUT,
)

head_state = torch.load(os.path.join(MODEL_DIR, "custom_heads.pt"), map_location="cpu")

print("Loading custom heads from:", os.path.join(MODEL_DIR, "custom_heads.pt"))
print("Head keys:", list(head_state.keys()))

model.tr_llm_head.load_state_dict(head_state["tr_llm_head"])
model.tr_feat_head.load_state_dict(head_state["tr_feat_head"])
model.tr_gate.load_state_dict(head_state["tr_gate"])

model.cc_llm_head.load_state_dict(head_state["cc_llm_head"])
model.cc_feat_head.load_state_dict(head_state["cc_feat_head"])
model.cc_gate.load_state_dict(head_state["cc_gate"])

model.lr_llm_head.load_state_dict(head_state["lr_llm_head"])
model.lr_feat_head.load_state_dict(head_state["lr_feat_head"])
model.lr_gate.load_state_dict(head_state["lr_gate"])

model.gra_llm_head.load_state_dict(head_state["gra_llm_head"])
model.gra_feat_head.load_state_dict(head_state["gra_feat_head"])
model.gra_gate.load_state_dict(head_state["gra_gate"])

model.to(DEVICE)
model.eval()

@torch.no_grad()
def predict_ielts(prompt: str, essay: str):
    tr_feat = extract_tr_features(prompt, essay)
    cc_feat = extract_cc_features(prompt, essay)
    lr_feat = extract_lr_features(prompt, essay)
    gra_feat = extract_gra_features(prompt, essay)

    tr_arr = standardize_features(tr_feat, TR_FEATURE_COLS, tr_feat_mean, tr_feat_std)
    cc_arr = standardize_features(cc_feat, CC_FEATURE_COLS, cc_feat_mean, cc_feat_std)
    lr_arr = standardize_features(lr_feat, LR_FEATURE_COLS, lr_feat_mean, lr_feat_std)
    gra_arr = standardize_features(gra_feat, GRA_FEATURE_COLS, gra_feat_mean, gra_feat_std)

    text = build_input_text(prompt, essay)
    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors="pt",
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        tr_features=torch.tensor(tr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
        cc_features=torch.tensor(cc_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
        lr_features=torch.tensor(lr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
        gra_features=torch.tensor(gra_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE),
    )

    preds = outputs["logits"].squeeze(0).detach().float().cpu().numpy()
    preds = np.clip(preds, SCORE_MIN, SCORE_MAX)

    raw_scores = {
        "TR": float(preds[0]),
        "CC": float(preds[1]),
        "LR": float(preds[2]),
        "GRA": float(preds[3]),
    }

    final_scores = {k: round_to_half(v) for k, v in raw_scores.items()}
    final_scores["Overall"] = round_to_half(np.mean([
        final_scores["TR"],
        final_scores["CC"],
        final_scores["LR"],
        final_scores["GRA"],
    ]))

    gate_info = {
        "tr_gate": float(outputs["tr_gate"].mean().detach().cpu()),
        "cc_gate": float(outputs["cc_gate"].mean().detach().cpu()),
        "lr_gate": float(outputs["lr_gate"].mean().detach().cpu()),
        "gra_gate": float(outputs["gra_gate"].mean().detach().cpu()),
    }

    return {
        "raw_scores": raw_scores,
        "final_scores": final_scores,
        "features": {
            "tr": tr_feat,
            "cc": cc_feat,
            "lr": lr_feat,
            "gra": gra_feat,
        },
        "gates": gate_info,
    }

def get_pred_scores(prompt, essay):
    res = predict_ielts(prompt, essay)
    return res, res["final_scores"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading custom heads from: /content/drive/MyDrive/B7_light_best_model/custom_heads.pt
Head keys: ['tr_llm_head', 'tr_feat_head', 'tr_gate', 'cc_llm_head', 'cc_feat_head', 'cc_gate', 'lr_llm_head', 'lr_feat_head', 'lr_gate', 'gra_llm_head', 'gra_feat_head', 'gra_gate']


In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
def embed_texts(texts, batch_size=32): return embedding_model.encode(texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def build_doc_text(row):
    return f"[PROMPT]\n{safe_text(row['prompt'])}\n\n[ESSAY]\n{safe_text(row['essay'])}".strip()
def build_retrieval_database(csv_path):
    df = pd.read_csv(csv_path)
    df = ensure_retrieval_columns(df)
    df["doc_text"] = df.apply(build_doc_text, axis=1)
    embeddings = embed_texts(df["doc_text"].tolist())
    return df, embeddings

In [10]:
# =========================
# QUALITY-AWARE RETRIEVAL HELPERS
# =========================

QUALITY_FEATURES = {
    "TR": [
        "tr_prompt_essay_sim",
        "tr_prompt_keyword_coverage",
        "tr_has_opinion",
        "tr_has_both_views",
        "tr_has_example",
        "tr_has_conclusion",
        "tr_word_count",
    ],
    "CC": [
        "cc_num_paragraphs",
        "cc_avg_paragraph_len",
        "cc_avg_sentence_len",
        "cc_sentence_len_std",
        "cc_discourse_marker_count",
        "cc_discourse_marker_diversity",
    ],
    "LR": [
        "lr_root_ttr",
        "lr_avg_word_len",
        "lr_long_word_ratio",
        "lr_repetition_ratio",
        "lr_unique_word_ratio",
        "lr_lexical_density_proxy",
    ],
    "GRA": [
        "gf_word_count",
        "gf_sentence_count",
        "gf_avg_sentence_len",
        "gf_short_sentence_ratio",
        "gf_long_sentence_ratio",
        "gf_punct_density",
        "gf_repeated_punct_ratio",
        "gf_lowercase_sent_start_ratio",
        "gf_lowercase_i_ratio",
        "gf_repeated_word_ratio",
        "gf_missing_terminal_punct",
    ],
}

QUALITY_INDEX = {}


def _quality_safe_float(x, default=0.0):
    try:
        if x is None:
            return float(default)
        if isinstance(x, str) and x.strip() == "":
            return float(default)
        return float(x)
    except Exception:
        return float(default)


def _quality_case_score(case, criterion, default=None):
    for key in [criterion, criterion.lower(), criterion.upper(), criterion.title()]:
        if key in case and case[key] is not None and str(case[key]).strip() != "":
            try:
                return float(case[key])
            except Exception:
                pass
    return default


def build_quality_profile_from_row(row, criterion):
    criterion = str(criterion).upper()
    prompt_type = str(row.get("prompt_type", "") or "")
    crit_score = _quality_case_score(row, criterion, 0.0)
    overall = _quality_case_score(row, "Overall", _quality_case_score(row, "overall", 0.0))

    parts = [
        f"criterion={criterion}",
        f"criterion_score={crit_score:.1f}",
        f"overall={overall:.1f}",
        f"prompt_type={prompt_type}",
    ]

    for col in QUALITY_FEATURES.get(criterion, []):
        parts.append(f"{col}={_quality_safe_float(row.get(col, 0.0)):.6f}")

    return " | ".join(parts)


def build_quality_query_payload(criterion, prompt, essay, state=None):
    criterion = str(criterion).upper()
    state = state or {}
    prompt_type = detect_prompt_type(prompt)
    pred_scores = dict(state.get("pred_scores") or {})

    if criterion == "TR":
        feats = extract_tr_features(prompt, essay)
    elif criterion == "CC":
        feats = extract_cc_features(prompt, essay)
    elif criterion == "LR":
        feats = extract_lr_features(prompt, essay)
    else:
        feats = extract_gra_features(prompt, essay)

    payload = {
        "criterion": criterion,
        "prompt_type": prompt_type,
        "criterion_score": _quality_safe_float(pred_scores.get(criterion, 0.0)),
    }
    for col in QUALITY_FEATURES.get(criterion, []):
        payload[col] = _quality_safe_float(feats.get(col, 0.0))
    return payload


def build_quality_query_text(criterion, prompt, essay, state=None):
    payload = build_quality_query_payload(criterion, prompt, essay, state)
    parts = []
    for k, v in payload.items():
        if isinstance(v, float):
            parts.append(f"{k}={v:.6f}")
        else:
            parts.append(f"{k}={v}")
    return " | ".join(parts), payload


def criterion_feature_distance(query_payload, case_row, criterion):
    criterion = str(criterion).upper()

    if criterion == "TR":
        return (
            abs(_quality_safe_float(query_payload.get("tr_prompt_essay_sim", 0.0)) - _quality_safe_float(case_row.get("tr_prompt_essay_sim", 0.0))) * 1.5
            + abs(_quality_safe_float(query_payload.get("tr_prompt_keyword_coverage", 0.0)) - _quality_safe_float(case_row.get("tr_prompt_keyword_coverage", 0.0))) * 1.8
            + abs(_quality_safe_float(query_payload.get("tr_has_opinion", 0.0)) - _quality_safe_float(case_row.get("tr_has_opinion", 0.0))) * 0.8
            + abs(_quality_safe_float(query_payload.get("tr_has_both_views", 0.0)) - _quality_safe_float(case_row.get("tr_has_both_views", 0.0))) * 0.8
            + abs(_quality_safe_float(query_payload.get("tr_has_example", 0.0)) - _quality_safe_float(case_row.get("tr_has_example", 0.0))) * 0.8
            + abs(_quality_safe_float(query_payload.get("tr_has_conclusion", 0.0)) - _quality_safe_float(case_row.get("tr_has_conclusion", 0.0))) * 0.8
            + abs(_quality_safe_float(query_payload.get("tr_word_count", 0.0)) - _quality_safe_float(case_row.get("tr_word_count", 0.0))) / 120.0
        )

    if criterion == "CC":
        return (
            abs(_quality_safe_float(query_payload.get("cc_num_paragraphs", 0.0)) - _quality_safe_float(case_row.get("cc_num_paragraphs", 0.0))) * 1.2
            + abs(_quality_safe_float(query_payload.get("cc_avg_paragraph_len", 0.0)) - _quality_safe_float(case_row.get("cc_avg_paragraph_len", 0.0))) / 60.0
            + abs(_quality_safe_float(query_payload.get("cc_avg_sentence_len", 0.0)) - _quality_safe_float(case_row.get("cc_avg_sentence_len", 0.0))) / 12.0
            + abs(_quality_safe_float(query_payload.get("cc_sentence_len_std", 0.0)) - _quality_safe_float(case_row.get("cc_sentence_len_std", 0.0))) / 10.0
            + abs(_quality_safe_float(query_payload.get("cc_discourse_marker_count", 0.0)) - _quality_safe_float(case_row.get("cc_discourse_marker_count", 0.0))) / 5.0
            + abs(_quality_safe_float(query_payload.get("cc_discourse_marker_diversity", 0.0)) - _quality_safe_float(case_row.get("cc_discourse_marker_diversity", 0.0))) * 1.5
        )

    if criterion == "LR":
        return (
            abs(_quality_safe_float(query_payload.get("lr_root_ttr", 0.0)) - _quality_safe_float(case_row.get("lr_root_ttr", 0.0))) / 1.5
            + abs(_quality_safe_float(query_payload.get("lr_avg_word_len", 0.0)) - _quality_safe_float(case_row.get("lr_avg_word_len", 0.0))) / 0.8
            + abs(_quality_safe_float(query_payload.get("lr_long_word_ratio", 0.0)) - _quality_safe_float(case_row.get("lr_long_word_ratio", 0.0))) * 6.0
            + abs(_quality_safe_float(query_payload.get("lr_repetition_ratio", 0.0)) - _quality_safe_float(case_row.get("lr_repetition_ratio", 0.0))) * 8.0
            + abs(_quality_safe_float(query_payload.get("lr_unique_word_ratio", 0.0)) - _quality_safe_float(case_row.get("lr_unique_word_ratio", 0.0))) * 6.0
            + abs(_quality_safe_float(query_payload.get("lr_lexical_density_proxy", 0.0)) - _quality_safe_float(case_row.get("lr_lexical_density_proxy", 0.0))) * 5.0
        )

    if criterion == "GRA":
        return (
            abs(_quality_safe_float(query_payload.get("gf_word_count", 0.0)) - _quality_safe_float(case_row.get("gf_word_count", 0.0))) / 120.0
            + abs(_quality_safe_float(query_payload.get("gf_sentence_count", 0.0)) - _quality_safe_float(case_row.get("gf_sentence_count", 0.0))) / 8.0
            + abs(_quality_safe_float(query_payload.get("gf_avg_sentence_len", 0.0)) - _quality_safe_float(case_row.get("gf_avg_sentence_len", 0.0))) / 10.0
            + abs(_quality_safe_float(query_payload.get("gf_short_sentence_ratio", 0.0)) - _quality_safe_float(case_row.get("gf_short_sentence_ratio", 0.0))) * 5.0
            + abs(_quality_safe_float(query_payload.get("gf_long_sentence_ratio", 0.0)) - _quality_safe_float(case_row.get("gf_long_sentence_ratio", 0.0))) * 5.0
            + abs(_quality_safe_float(query_payload.get("gf_repeated_punct_ratio", 0.0)) - _quality_safe_float(case_row.get("gf_repeated_punct_ratio", 0.0))) * 8.0
            + abs(_quality_safe_float(query_payload.get("gf_lowercase_sent_start_ratio", 0.0)) - _quality_safe_float(case_row.get("gf_lowercase_sent_start_ratio", 0.0))) * 10.0
            + abs(_quality_safe_float(query_payload.get("gf_lowercase_i_ratio", 0.0)) - _quality_safe_float(case_row.get("gf_lowercase_i_ratio", 0.0))) * 10.0
            + abs(_quality_safe_float(query_payload.get("gf_repeated_word_ratio", 0.0)) - _quality_safe_float(case_row.get("gf_repeated_word_ratio", 0.0))) * 8.0
            + abs(_quality_safe_float(query_payload.get("gf_missing_terminal_punct", 0.0)) - _quality_safe_float(case_row.get("gf_missing_terminal_punct", 0.0))) * 2.0
        )

    return 0.0


def build_quality_index(df, embed_model, store_name="main_quality", criteria=("TR", "CC", "LR", "GRA")):
    df = df.copy().reset_index(drop=True)

    if "prompt_type" not in df.columns:
        df["prompt_type"] = df["prompt"].fillna("").map(detect_prompt_type)

    QUALITY_INDEX[store_name] = {}

    for criterion in criteria:
        texts = [build_quality_profile_from_row(row, criterion) for _, row in df.iterrows()]
        embs = embed_model.encode(
            texts,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        QUALITY_INDEX[store_name][criterion] = {
            "data": df,
            "texts": texts,
            "embeddings": embs.astype(np.float32),
        }
        print(f"[quality-index] {criterion}: {len(texts)} cases")

    return QUALITY_INDEX[store_name]


def retrieve_quality_cases_for_criterion(
    criterion,
    prompt,
    essay,
    state,
    store_name="main_quality",
    top_k_stage1=40,
    top_k_final=5,
    target_before=None,
    allowed_band_gap=1.0,
):
    criterion = str(criterion).upper()

    if store_name not in QUALITY_INDEX or criterion not in QUALITY_INDEX[store_name]:
        return []

    bucket = QUALITY_INDEX[store_name][criterion]
    data = bucket["data"]
    emb = bucket["embeddings"]

    query_text, query_payload = build_quality_query_text(criterion, prompt, essay, state)
    q_emb = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)[0]

    sims = emb @ q_emb
    stage1_idx = np.argsort(-sims)[:max(top_k_stage1, top_k_final)]

    pred_scores = dict(state.get("pred_scores") or {})
    if target_before is None:
        target_before = _quality_safe_float(pred_scores.get(criterion, 0.0))

    min_band = max(4.0, float(target_before) - float(allowed_band_gap))
    prompt_type = detect_prompt_type(prompt)

    candidates = []
    for idx in stage1_idx:
        row = data.iloc[int(idx)].to_dict()
        row["_vector_sim"] = float(sims[int(idx)])
        row["_criterion"] = criterion
        row["_idx"] = int(idx)

        case_band = _quality_case_score(row, criterion, 0.0)
        if case_band < min_band:
            continue

        band_match = max(0.0, 1.0 - abs(case_band - float(target_before)) / 2.0)
        feat_dist = criterion_feature_distance(query_payload, row, criterion)
        feature_match = 1.0 / (1.0 + feat_dist)
        ptype_bonus = 0.05 if str(row.get("prompt_type", "")) == prompt_type else 0.0

        final_score = (0.55 * float(row["_vector_sim"])) + (0.30 * float(band_match)) + (0.15 * float(feature_match)) + ptype_bonus

        row["_band_match"] = float(band_match)
        row["_feature_distance"] = float(feat_dist)
        row["_feature_match"] = float(feature_match)
        row["_final_quality_score"] = float(final_score)
        candidates.append(row)

    if len(candidates) < max(3, top_k_final):
        candidates = []
        for idx in stage1_idx:
            row = data.iloc[int(idx)].to_dict()
            row["_vector_sim"] = float(sims[int(idx)])
            row["_criterion"] = criterion
            row["_idx"] = int(idx)

            case_band = _quality_case_score(row, criterion, 0.0)
            band_match = max(0.0, 1.0 - abs(case_band - float(target_before)) / 2.0)
            feat_dist = criterion_feature_distance(query_payload, row, criterion)
            feature_match = 1.0 / (1.0 + feat_dist)
            ptype_bonus = 0.05 if str(row.get("prompt_type", "")) == prompt_type else 0.0
            final_score = (0.55 * float(row["_vector_sim"])) + (0.30 * float(band_match)) + (0.15 * float(feature_match)) + ptype_bonus

            row["_band_match"] = float(band_match)
            row["_feature_distance"] = float(feat_dist)
            row["_feature_match"] = float(feature_match)
            row["_final_quality_score"] = float(final_score)
            candidates.append(row)

    candidates = sorted(candidates, key=lambda x: x["_final_quality_score"], reverse=True)[:top_k_final]
    state.setdefault("quality_cases", {})
    state["quality_cases"][criterion] = candidates
    return candidates

In [11]:
EXPLAIN_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
explain_tokenizer = AutoTokenizer.from_pretrained(EXPLAIN_MODEL_NAME, use_fast=False)
if explain_tokenizer.pad_token is None: explain_tokenizer.pad_token = explain_tokenizer.eos_token
explain_tokenizer.padding_side = "left"
explain_model = AutoModelForCausalLM.from_pretrained(EXPLAIN_MODEL_NAME, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto").eval()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [12]:
def normalize_diff(a, b, scale):
    try:
        return abs(float(a) - float(b)) / float(scale)
    except Exception:
        return 0.0


def detect_prompt_type(prompt: str):
    p = normalize_text(prompt)

    if "advantages outweigh the disadvantages" in p or "advantages and disadvantages" in p:
        return "advantages_disadvantages"
    if "discuss both views" in p and "give your own opinion" in p:
        return "both_views_opinion"
    if "discuss both views" in p:
        return "both_views"
    if "to what extent do you agree or disagree" in p:
        return "agree_disagree"
    if "what are the causes" in p and "what can be done" in p:
        return "causes_solutions"
    if "what problems" in p and "what solutions" in p:
        return "problems_solutions"
    if "what are the reasons" in p and "what are the effects" in p:
        return "reasons_effects"
    return "other"


def extract_retrieval_features(prompt, essay):
    text = safe_text(essay)
    prompt_text = safe_text(prompt)

    words = re.findall(r"\b\w+\b", text.lower())
    prompt_words = re.findall(r"\b\w+\b", prompt_text.lower())

    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    essay_len = len(words)
    unique_words = len(set(words))
    lexical_diversity = unique_words / max(essay_len, 1)
    avg_sent_len = essay_len / max(len(sentences), 1)

    prompt_relevance = 0.0
    try:
        tr_feat = extract_tr_features(prompt, essay)
        if "prompt_relevance" in tr_feat:
            prompt_relevance = float(tr_feat["prompt_relevance"])
        else:
            prompt_set = set([w for w in prompt_words if w not in STOPWORDS])
            essay_set = set([w for w in words if w not in STOPWORDS])
            if prompt_set:
                prompt_relevance = len(prompt_set & essay_set) / len(prompt_set)
    except Exception:
        prompt_set = set([w for w in prompt_words if w not in STOPWORDS])
        essay_set = set([w for w in words if w not in STOPWORDS])
        if prompt_set:
            prompt_relevance = len(prompt_set & essay_set) / len(prompt_set)

    return {
        "essay_len": float(essay_len),
        "prompt_relevance": float(prompt_relevance),
        "lexical_diversity": float(lexical_diversity),
        "readability_score": float(avg_sent_len),
    }


def ensure_retrieval_columns(df):
    df = df.copy()

    if "evaluation" not in df.columns:
        df["evaluation"] = ""

    if "essay_len" not in df.columns:
        df["essay_len"] = df["essay"].astype(str).apply(lambda x: len(re.findall(r"\b\w+\b", x)))

    if "prompt_relevance" not in df.columns:
        def _prompt_rel(row):
            try:
                return float(jaccard_coverage(row["prompt"], row["essay"]))
            except Exception:
                return 0.0
        df["prompt_relevance"] = df.apply(_prompt_rel, axis=1)

    if "lexical_diversity" not in df.columns:
        def _lex_div(x):
            words = re.findall(r"\b\w+\b", str(x).lower())
            return len(set(words)) / max(len(words), 1)
        df["lexical_diversity"] = df["essay"].astype(str).apply(_lex_div)

    if "readability_score" not in df.columns:
        def _readability(x):
            sents = [s.strip() for s in re.split(r"[.!?]+", str(x)) if s.strip()]
            words = re.findall(r"\b\w+\b", str(x).lower())
            return len(words) / max(len(sents), 1)
        df["readability_score"] = df["essay"].astype(str).apply(_readability)

    if "prompt_type" not in df.columns:
        df["prompt_type"] = df["prompt"].astype(str).apply(detect_prompt_type)

    return df


def quality_distance(row, pred_scores, feat_dict):
    score_dist = (
        normalize_diff(row.get("TR", 0), pred_scores["TR"], 9.0) +
        normalize_diff(row.get("CC", 0), pred_scores["CC"], 9.0) +
        normalize_diff(row.get("LR", 0), pred_scores["LR"], 9.0) +
        normalize_diff(row.get("GRA", 0), pred_scores["GRA"], 9.0)
    )

    feat_dist = (
        normalize_diff(row.get("essay_len", 0), feat_dict["essay_len"], 400.0) +
        normalize_diff(row.get("prompt_relevance", 0), feat_dict["prompt_relevance"], 1.0) +
        normalize_diff(row.get("lexical_diversity", 0), feat_dict["lexical_diversity"], 1.0) +
        normalize_diff(row.get("readability_score", 0), feat_dict["readability_score"], 100.0)
    )

    return score_dist + 0.5 * feat_dist


def retrieve_cases(
    query_embedding,
    db_embeddings,
    df,
    pred_scores,
    feat_dict,
    prompt,
    top_k_vector=20,
    top_k_final=5,
):
    df = ensure_retrieval_columns(df)

    sims = cosine_similarity(query_embedding.reshape(1, -1), db_embeddings)[0]
    candidate_idx = np.argsort(-sims)[:top_k_vector]
    candidates = df.iloc[candidate_idx].copy()

    candidates["vector_sim"] = sims[candidate_idx]
    candidates["quality_dist"] = candidates.apply(
        lambda row: quality_distance(row, pred_scores, feat_dict),
        axis=1
    )

    query_prompt_type = detect_prompt_type(prompt)
    candidates["same_prompt_type"] = (
        candidates["prompt_type"].astype(str) == str(query_prompt_type)
    ).astype(float)

    # Hybrid score: semantic + score/profile similarity + prompt-type bonus
    candidates["final_score"] = (
        candidates["vector_sim"]
        - 0.70 * candidates["quality_dist"]
        + 0.05 * candidates["same_prompt_type"]
    )

    candidates = candidates.sort_values("final_score", ascending=False)
    return candidates.head(top_k_final)


def retrieve_similar_essays_for_inference(
    prompt,
    essay,
    df,
    db_embeddings,
    top_k_vector=20,
    top_k_final=5,
    pred_result=None,
    pred_scores=None,
):
    if pred_scores is None:
        pred_result, pred_scores = get_pred_scores(prompt, essay)

    feat_dict = extract_retrieval_features(prompt, essay)

    query_text = f"""
IELTS Writing Task 2 Prompt:
{prompt}

Essay:
{essay}

Predicted scores:
TR={pred_scores['TR']}, CC={pred_scores['CC']}, LR={pred_scores['LR']}, GRA={pred_scores['GRA']}
""".strip()

    query_embedding = embed_texts([query_text], batch_size=1)[0]

    top_cases_df = retrieve_cases(
        query_embedding=query_embedding,
        db_embeddings=db_embeddings,
        df=df,
        pred_scores=pred_scores,
        feat_dict=feat_dict,
        prompt=prompt,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
    )

    top_cases = top_cases_df.to_dict(orient="records")

    return {
        "pred_result": pred_result,
        "pred_scores": pred_scores,
        "feat_dict": feat_dict,
        "top_cases": top_cases,
    }


def format_top_cases_for_prompt(top_cases, prompt=None, max_cases=3, max_chars_each=900):
    if top_cases is None:
        return "[]"

    try:
        df_cases = pd.DataFrame(top_cases)
    except Exception:
        return "[]"

    if len(df_cases) == 0:
        return "[]"

    if prompt is not None and "prompt" in df_cases.columns:
        q_type = detect_prompt_type(prompt)
        df_cases["prompt_type"] = df_cases["prompt"].astype(str).apply(detect_prompt_type)
        matched = df_cases[df_cases["prompt_type"] == q_type]
        if len(matched) > 0:
            df_cases = matched.copy()

    if "final_score" in df_cases.columns:
        df_cases = df_cases.sort_values("final_score", ascending=False)

    items = []
    for i, (_, row) in enumerate(df_cases.head(max_cases).iterrows(), start=1):
        prompt_txt = str(row.get("prompt", ""))[:400]
        essay_txt = str(row.get("essay", ""))[:max_chars_each]

        eval_txt = str(row.get("evaluation", "")).strip()
        if eval_txt:
            eval_txt = re.sub(r"\s+", " ", eval_txt)[:300]

        tr = row.get("TR", row.get("tr", "N/A"))
        cc = row.get("CC", row.get("cc", "N/A"))
        lr = row.get("LR", row.get("lr", "N/A"))
        gra = row.get("GRA", row.get("gra", "N/A"))
        overall = row.get("Overall", row.get("overall", row.get("band", "N/A")))
        score = row.get("final_score", row.get("vector_sim", "N/A"))

        block = f"""
[Reference Case {i}]
Prompt: {prompt_txt}
Scores: TR={tr}, CC={cc}, LR={lr}, GRA={gra}, Overall={overall}
Hybrid score: {score}
Essay Snippet:
{essay_txt}
""".strip()

        if eval_txt:
            block += f"\nReference Note: {eval_txt}"

        items.append(block)

    return "\n\n".join(items) if items else "[]"

In [13]:
def mistral_explain_writer(prompt_text, temperature=0.15, max_new_tokens=600):
    full_prompt = f"[INST]\nYou are an expert IELTS examiner.\n{prompt_text}\n[/INST]"
    model_inputs = explain_tokenizer([full_prompt], return_tensors="pt", truncation=True, max_length=4096).to(explain_model.device)
    gen_kwargs = {"max_new_tokens": max_new_tokens, "repetition_penalty": 1.05}
    if temperature > 0: gen_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": 0.9})
    with torch.no_grad():
        output_ids = explain_model.generate(**model_inputs, **gen_kwargs)
    return explain_tokenizer.batch_decode(output_ids[:, model_inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def safe_json_loads(text, default=None):
    if default is None:
        default = {}
    if text is None:
        return default

    text = str(text).strip()
    if not text:
        return default

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    m = re.search(r"(\{.*\}|\[.*\])", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    return default

In [14]:
def tool_predict_scores(prompt, essay, state):
    res, scores = get_pred_scores(prompt, essay)
    state["pred_scores"] = scores
    return {"scores": scores}

tool_detect_task_mismatch

In [15]:
def build_task_mismatch_prompt(prompt, essay, pred_scores):
    return f"""
You are a careful IELTS Writing Task 2 task-response checker.

Your job:
Determine whether the essay clearly addresses the prompt, only partially addresses it, or is off-topic.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Judging rules:
1. Focus only on task alignment and relevance.
2. Do NOT judge grammar, vocabulary, or cohesion here.
3. "pass" means the essay clearly answers the prompt overall, even if development is not perfect.
4. "partial_mismatch" should be used only if one important task part is clearly missing, seriously misunderstood, or replaced by another idea.
5. "off_topic" should be used only if the essay mainly answers a different question or is largely irrelevant.
6. Do NOT mark partial_mismatch merely because the essay is somewhat general, not deep enough, or lacks strong examples.
7. If the essay keeps the correct topic, correct task type, and a clear overall position, prefer "pass".
8. Be conservative: if uncertain between "pass" and "partial_mismatch", choose "pass".

Output valid JSON only in this format:
{{
  "verdict": "pass" or "partial_mismatch" or "off_topic",
  "reason": "...",
  "missing_parts": ["...", "..."],
  "evidence": "..."
}}
""".strip()


def normalize_task_check_result(obj):
    if not isinstance(obj, dict):
        return {
            "verdict": "pass",
            "reason": "",
            "missing_parts": [],
            "evidence": "",
        }

    verdict = str(obj.get("verdict", "pass")).strip().lower()
    if verdict not in ["pass", "partial_mismatch", "off_topic"]:
        verdict = "pass"

    missing_parts = obj.get("missing_parts", [])
    if not isinstance(missing_parts, list):
        missing_parts = []

    return {
        "verdict": verdict,
        "reason": str(obj.get("reason", "")).strip(),
        "missing_parts": [str(x).strip() for x in missing_parts if str(x).strip()],
        "evidence": str(obj.get("evidence", "")).strip(),
    }


def round_to_half_band(x):
    return round(float(x) * 2) / 2


def apply_tr_guardrail(pred_scores, task_check):
    if pred_scores is None:
        return pred_scores

    guarded = dict(pred_scores)
    verdict = str(task_check.get("verdict", "pass")).strip().lower()

    old_tr = float(guarded["TR"])

    if verdict == "pass":
        new_tr = old_tr
    elif verdict == "partial_mismatch":
        new_tr = min(old_tr, 6.5)
    elif verdict == "off_topic":
        new_tr = min(old_tr, 5.5)
    else:
        new_tr = old_tr

    guarded["TR"] = round_to_half_band(new_tr)
    overall = np.mean([
        guarded["TR"],
        guarded["CC"],
        guarded["LR"],
        guarded["GRA"],
    ])
    guarded["Overall"] = round_to_half_band(overall)

    return guarded


def tool_detect_task_mismatch(prompt, essay, state):
    pred_scores = state["pred_scores"]

    task_prompt = build_task_mismatch_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
    )

    raw = mistral_explain_writer(
        task_prompt,
        temperature=0.0,
        max_new_tokens=500,
    )

    parsed = safe_json_loads(raw, default={})
    task_check = normalize_task_check_result(parsed)
    task_check["raw"] = str(raw)

    state["task_check"] = task_check

    original_scores = dict(state["pred_scores"])
    adjusted_scores = apply_tr_guardrail(state["pred_scores"], task_check)
    state["pred_scores"] = adjusted_scores

    return {
        "task_check": {
            "verdict": task_check["verdict"],
            "reason": task_check["reason"],
            "missing_parts": task_check["missing_parts"],
            "evidence": task_check["evidence"],
        },
        "score_adjustment": {
            "TR_before": original_scores["TR"],
            "TR_after": adjusted_scores["TR"],
            "Overall_before": original_scores["Overall"],
            "Overall_after": adjusted_scores["Overall"],
        }
    }

In [16]:

# =========================
# SCORE GUARDRAIL TOOLS
# =========================

def recompute_overall_from_scores(scores):
    fixed = dict(scores or {})
    for crit in ["TR", "CC", "LR", "GRA"]:
        fixed[crit] = round_to_half_band(float(fixed.get(crit, 0.0)))
    fixed["Overall"] = round_to_half_band(np.mean([fixed["TR"], fixed["CC"], fixed["LR"], fixed["GRA"]]))
    return fixed


def apply_band_cap(pred_scores, criterion, max_band):
    if pred_scores is None:
        return pred_scores

    adjusted = dict(pred_scores)
    old_val = float(adjusted.get(criterion, 0.0))
    new_val = min(old_val, float(max_band))
    adjusted[criterion] = round_to_half_band(new_val)
    return recompute_overall_from_scores(adjusted)


def _content_words_for_lr(essay):
    words = [str(w).lower() for w in tokenize_words(essay)]
    return [w for w in words if len(w) >= 4 and w not in STOPWORDS]


def _top_repeated_terms(essay, min_count=3, top_n=5):
    content_words = _content_words_for_lr(essay)
    freq = Counter(content_words)
    ranked = sorted([(w, c) for w, c in freq.items() if c >= min_count], key=lambda x: (-x[1], x[0]))
    return ranked[:top_n]


def tool_check_argument_development(prompt, essay, state):
    pred_scores = dict(state.get("pred_scores") or {})
    tr_before = float(pred_scores.get("TR", 0.0))

    feats = extract_tr_features(prompt, essay)
    prompt_type = detect_prompt_type(prompt)
    paras = split_paragraphs(essay)

    issues = []
    evidence = []

    coverage = float(feats.get("tr_prompt_keyword_coverage", 0.0))
    sim = float(feats.get("tr_prompt_essay_sim", 0.0))
    has_opinion_flag = float(feats.get("tr_has_opinion", 0.0)) >= 0.5
    has_both_views_flag = float(feats.get("tr_has_both_views", 0.0)) >= 0.5
    has_example_flag = float(feats.get("tr_has_example", 0.0)) >= 0.5
    has_conclusion_flag = float(feats.get("tr_has_conclusion", 0.0)) >= 0.5
    word_cnt = float(feats.get("tr_word_count", 0.0))

    lower_essay = str(essay).lower()
    intro_like = 1 if any(x in lower_essay[:350] for x in ["in this essay", "this essay", "i believe", "i think", "it is argued", "while"]) else 0
    conclusion_like = 1 if any(x in lower_essay[-350:] for x in ["in conclusion", "to conclude", "to sum up", "overall"]) else 0

    severe_points = 0
    moderate_points = 0

    # prompt-type requirements: nhẹ tay hơn, không cap nặng chỉ vì thiếu marker
    if prompt_type in ["both_views", "both_views_opinion"] and not has_both_views_flag:
        issues.append("The essay may not fully cover both required views.")
        evidence.append("Both-view markers were not detected strongly.")
        severe_points += 1

    if prompt_type == "agree_disagree" and not has_opinion_flag:
        issues.append("The writer's opinion is not stated clearly enough.")
        evidence.append("No strong opinion marker was detected.")
        severe_points += 1

    # prompt alignment: cần khá yếu mới tính là severe
    if coverage < 0.07 and sim < 0.26 and tr_before >= 6.5:
        issues.append("Prompt coverage looks weak, so the response may be only partially aligned with the task.")
        evidence.append(f"prompt_keyword_coverage={coverage:.2f}, prompt_essay_similarity={sim:.2f}")
        severe_points += 2
    elif coverage < 0.10 and sim < 0.31 and tr_before >= 7.0:
        issues.append("Task coverage looks a bit weaker than the current TR score suggests.")
        evidence.append(f"prompt_keyword_coverage={coverage:.2f}, prompt_essay_similarity={sim:.2f}")
        moderate_points += 1

    # length: bớt gắt
    if word_cnt < 195 and tr_before >= 7.0:
        issues.append("The response is rather short, which may limit idea development.")
        evidence.append(f"word_count={int(word_cnt)}")
        moderate_points += 2
    elif word_cnt < 225 and tr_before >= 7.5:
        issues.append("The response length is slightly limited for a very strong TR band.")
        evidence.append(f"word_count={int(word_cnt)}")
        moderate_points += 1

    # example / conclusion: chỉ là moderate, không kéo quá mạnh
    if not has_example_flag and tr_before >= 7.0:
        issues.append("The essay could use a clearer supporting example.")
        evidence.append("No strong example marker was detected.")
        moderate_points += 1

    if not has_conclusion_flag and tr_before >= 7.0:
        issues.append("The response does not show a very clear concluding signal.")
        evidence.append("A conclusion marker was not detected near the end.")
        moderate_points += 1

    if len(paras) < 3 and tr_before >= 7.0:
        issues.append("Limited paragraph separation may weaken development.")
        evidence.append(f"num_paragraphs={len(paras)}")
        moderate_points += 1

    if intro_like == 0 and tr_before >= 7.5:
        issues.append("The essay opening is not strongly framed as a task response.")
        evidence.append("No strong introductory signal was detected.")
        moderate_points += 1

    if conclusion_like == 0 and tr_before >= 7.5:
        issues.append("The ending is not strongly shaped as a full response conclusion.")
        evidence.append("No strong concluding signal was detected.")
        moderate_points += 1

    total_points = severe_points * 2 + moderate_points

    recommended_cap = tr_before
    risk_level = "low"

    # nặng tay hơn một nấc
    if severe_points >= 2 and total_points >= 5:
        recommended_cap = min(tr_before, 6.0)
        risk_level = "high"
    elif (severe_points >= 1 and total_points >= 4) or (total_points >= 5 and tr_before >= 7.0):
        recommended_cap = min(tr_before, 6.5)
        risk_level = "high"
    elif total_points >= 3 and tr_before >= 7.0:
        recommended_cap = min(tr_before, 7.0)
        risk_level = "medium"
    elif total_points >= 2 and tr_before >= 7.5:
        recommended_cap = min(tr_before, 7.0)
        risk_level = "medium"
    elif total_points >= 1 and tr_before >= 8.0:
        recommended_cap = min(tr_before, 7.5)
        risk_level = "medium"

    tr_after = tr_before
    overall_after = float(pred_scores.get("Overall", 0.0))

    if recommended_cap < tr_before:
        adjusted_scores = apply_band_cap(pred_scores, "TR", recommended_cap)
        state["pred_scores"] = adjusted_scores
        tr_after = float(adjusted_scores.get("TR", tr_before))
        overall_after = float(adjusted_scores.get("Overall", pred_scores.get("Overall", 0.0)))
    else:
        tr_after = float(pred_scores.get("TR", tr_before))
        overall_after = float(pred_scores.get("Overall", 0.0))

    result = {
        "criterion": "TR",
        "checked": True,
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "recommended_cap": round_to_half_band(recommended_cap),
        "score_before": tr_before,
        "score_after": tr_after,
    }
    state.setdefault("score_checks", {})["TR"] = result

    return {
        "criterion": "TR",
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "score_adjustment": {
            "TR_before": tr_before,
            "TR_after": tr_after,
            "Overall_after": overall_after,
        },
    }
def tool_check_coherence_risk(prompt, essay, state):
    pred_scores = dict(state.get("pred_scores") or {})
    cc_before = float(pred_scores.get("CC", 0.0))

    feats = extract_cc_features(prompt, essay)
    paras = split_paragraphs(essay)
    lower_essay = str(essay).lower()

    issues = []
    evidence = []

    num_paragraphs = float(feats.get("cc_num_paragraphs", len(paras)))
    avg_para_len = float(feats.get("cc_avg_paragraph_len", 0.0))
    avg_sent_len = float(feats.get("cc_avg_sentence_len", 0.0))
    sent_len_std = float(feats.get("cc_sentence_len_std", 0.0))
    marker_count = float(feats.get("cc_discourse_marker_count", 0.0))
    marker_div = float(feats.get("cc_discourse_marker_diversity", 0.0))

    repeated_linkers = 0
    for linker in ["firstly", "secondly", "thirdly", "moreover", "besides", "however", "therefore", "in conclusion"]:
        if lower_essay.count(linker) >= 2:
            repeated_linkers += 1

    severe_points = 0
    moderate_points = 0

    if num_paragraphs < 3 and cc_before >= 7.0:
        issues.append("Paragraphing is a bit limited for a very high CC score.")
        evidence.append(f"cc_num_paragraphs={num_paragraphs:.0f}")
        moderate_points += 2

    if avg_para_len > 155 and cc_before >= 7.0:
        issues.append("Paragraphs look quite long and block-like.")
        evidence.append(f"cc_avg_paragraph_len={avg_para_len:.1f}")
        moderate_points += 1

    if avg_para_len < 40 and num_paragraphs >= 4 and cc_before >= 7.5:
        issues.append("Paragraphing may look slightly fragmented.")
        evidence.append(f"cc_avg_paragraph_len={avg_para_len:.1f}")
        moderate_points += 1

    if avg_sent_len > 30 and cc_before >= 7.0:
        issues.append("Sentences are rather long on average, which may reduce clarity.")
        evidence.append(f"cc_avg_sentence_len={avg_sent_len:.1f}")
        moderate_points += 1

    if sent_len_std > 14 and cc_before >= 7.5:
        issues.append("Sentence length varies quite a lot, which may affect progression control.")
        evidence.append(f"cc_sentence_len_std={sent_len_std:.1f}")
        moderate_points += 1

    if marker_count <= 2 and cc_before >= 7.5:
        issues.append("The essay may not show enough explicit linking support for a very strong CC band.")
        evidence.append(f"cc_discourse_marker_count={marker_count:.0f}")
        moderate_points += 1

    if marker_div < 0.40 and cc_before >= 7.5:
        issues.append("Linking appears somewhat repetitive.")
        evidence.append(f"cc_discourse_marker_diversity={marker_div:.2f}")
        moderate_points += 1

    if repeated_linkers >= 3 and cc_before >= 7.0:
        issues.append("A small set of linkers seems to be reused too heavily.")
        evidence.append(f"repeated_linker_types={repeated_linkers}")
        moderate_points += 1

    if num_paragraphs == 3 and avg_para_len > 125 and cc_before >= 7.5:
        issues.append("The body structure may be a little compressed for a very high CC score.")
        evidence.append(f"cc_num_paragraphs={num_paragraphs:.0f}, cc_avg_paragraph_len={avg_para_len:.1f}")
        moderate_points += 1

    total_points = severe_points * 2 + moderate_points

    recommended_cap = cc_before
    risk_level = "low"

    if total_points >= 5 and cc_before >= 7.0:
        recommended_cap = min(cc_before, 6.5)
        risk_level = "high"
    elif total_points >= 4 and cc_before >= 7.0:
        recommended_cap = min(cc_before, 7.0)
        risk_level = "medium"
    elif total_points >= 2 and cc_before >= 7.5:
        recommended_cap = min(cc_before, 7.0)
        risk_level = "medium"
    elif total_points >= 1 and cc_before >= 8.0:
        recommended_cap = min(cc_before, 7.5)
        risk_level = "medium"

    cc_after = cc_before
    overall_after = float(pred_scores.get("Overall", 0.0))

    if recommended_cap < cc_before:
        adjusted_scores = apply_band_cap(pred_scores, "CC", recommended_cap)
        state["pred_scores"] = adjusted_scores
        cc_after = float(adjusted_scores.get("CC", cc_before))
        overall_after = float(adjusted_scores.get("Overall", pred_scores.get("Overall", 0.0)))
    else:
        cc_after = float(pred_scores.get("CC", cc_before))
        overall_after = float(pred_scores.get("Overall", 0.0))

    result = {
        "criterion": "CC",
        "checked": True,
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "recommended_cap": round_to_half_band(recommended_cap),
        "score_before": cc_before,
        "score_after": cc_after,
    }
    state.setdefault("score_checks", {})["CC"] = result

    return {
        "criterion": "CC",
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "score_adjustment": {
            "CC_before": cc_before,
            "CC_after": cc_after,
            "Overall_after": overall_after,
        },
    }

def tool_check_lexical_risk(prompt, essay, state):
    pred_scores = dict(state.get("pred_scores") or {})
    lr_before = float(pred_scores.get("LR", 0.0))
    feats = extract_lr_features(prompt, essay)
    repeated_terms = _top_repeated_terms(essay, min_count=3, top_n=5)

    repetition_ratio_val = float(feats.get("lr_repetition_ratio", 0.0))
    unique_ratio_val = float(feats.get("lr_unique_word_ratio", 0.0))
    long_word_ratio_val = float(feats.get("lr_long_word_ratio", 0.0))
    lexical_density_val = float(feats.get("lr_lexical_density_proxy", 0.0))
    root_ttr_val = float(feats.get("lr_root_ttr", 0.0))
    avg_word_len_val = float(feats.get("lr_avg_word_len", 0.0))

    issues = []
    evidence = []

    severe_points = 0
    moderate_points = 0

    # chỉ gắt khi repetition thực sự cao
    if repetition_ratio_val >= 0.19 and unique_ratio_val < 0.40 and lr_before >= 7.0:
        issues.append("Word repetition is quite high, so the lexical range may be overestimated.")
        evidence.append(f"lr_repetition_ratio={repetition_ratio_val:.2f}, lr_unique_word_ratio={unique_ratio_val:.2f}")
        severe_points += 1
    elif repetition_ratio_val >= 0.14 and lr_before >= 7.5:
        issues.append("Repeated wording may be inflating LR slightly.")
        evidence.append(f"lr_repetition_ratio={repetition_ratio_val:.2f}")
        moderate_points += 1

    if unique_ratio_val < 0.40 and lr_before >= 7.5:
        issues.append("The vocabulary range looks a bit limited for a very high LR prediction.")
        evidence.append(f"lr_unique_word_ratio={unique_ratio_val:.2f}")
        moderate_points += 1
    elif unique_ratio_val < 0.36 and lr_before >= 7.0:
        issues.append("The vocabulary set looks rather limited.")
        evidence.append(f"lr_unique_word_ratio={unique_ratio_val:.2f}")
        severe_points += 1

    if len(repeated_terms) >= 4 and lr_before >= 7.5:
        repeated_text = ", ".join([f"`{w}` x{c}" for w, c in repeated_terms])
        issues.append("Several content words are reused many times.")
        evidence.append(f"Top repeated terms: {repeated_text}")
        moderate_points += 1

    # sophistication: nhẹ hơn nhiều
    if long_word_ratio_val < 0.15 and lexical_density_val < 0.49 and root_ttr_val < 3.8 and lr_before >= 8.0:
        issues.append("The essay does not show enough lexical sophistication for a very high LR score.")
        evidence.append(
            f"lr_long_word_ratio={long_word_ratio_val:.2f}, "
            f"lr_lexical_density_proxy={lexical_density_val:.2f}, "
            f"lr_root_ttr={root_ttr_val:.2f}"
        )
        moderate_points += 2
    elif avg_word_len_val < 4.2 and lr_before >= 8.0:
        issues.append("Vocabulary sophistication looks slightly modest for a very high LR band.")
        evidence.append(f"lr_avg_word_len={avg_word_len_val:.2f}")
        moderate_points += 1

    total_points = severe_points * 2 + moderate_points

    recommended_cap = lr_before
    risk_level = "low"

    if severe_points >= 2 or (severe_points >= 1 and total_points >= 4):
        recommended_cap = min(lr_before, 6.0)
        risk_level = "high"
    elif total_points >= 4 and lr_before >= 7.0:
        recommended_cap = min(lr_before, 6.5)
        risk_level = "high"
    elif total_points >= 3 and lr_before >= 7.0:
        recommended_cap = min(lr_before, 7.0)
        risk_level = "medium"
    elif total_points >= 2 and lr_before >= 7.5:
        recommended_cap = min(lr_before, 7.0)
        risk_level = "medium"
    elif total_points >= 1 and lr_before >= 8.0:
        recommended_cap = min(lr_before, 7.5)
        risk_level = "medium"

    adjusted_scores = dict(pred_scores)
    if recommended_cap < lr_before:
        adjusted_scores = apply_band_cap(pred_scores, "LR", recommended_cap)
        state["pred_scores"] = adjusted_scores

    result = {
        "criterion": "LR",
        "checked": True,
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "recommended_cap": round_to_half_band(recommended_cap),
        "score_before": lr_before,
        "score_after": float((state.get("pred_scores") or pred_scores).get("LR", lr_before)),
    }
    state.setdefault("score_checks", {})["LR"] = result

    return {
        "criterion": "LR",
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "score_adjustment": {
            "LR_before": lr_before,
            "LR_after": float((state.get("pred_scores") or pred_scores).get("LR", lr_before)),
            "Overall_after": float((state.get("pred_scores") or pred_scores).get("Overall", pred_scores.get("Overall", 0.0))),
        },
    }


def tool_check_grammar_risk(prompt, essay, state):
    pred_scores = dict(state.get("pred_scores") or {})
    gra_before = float(pred_scores.get("GRA", 0.0))
    gra_feats = extract_gra_features(prompt, essay)
    candidates = extract_gra_error_candidates_v2(essay, max_candidates=8, max_per_type=3)

    severe_errors = [c for c in candidates if int(c.get("severity", 1)) >= 3]
    issues = []
    evidence = []

    severe_points = 0
    moderate_points = 0

    # bớt gắt: phải có nhiều lỗi nặng hơn mới cap sâu
    if len(severe_errors) >= 3 and gra_before >= 7.0:
        issues.append("Several visible grammar errors suggest that GRA may be overestimated.")
        evidence.append("Severe grammar signals: " + "; ".join([f"`{c.get('snippet', '')}` ({c.get('reason', '')})" for c in severe_errors[:3]]))
        severe_points += 2
    elif len(severe_errors) >= 2 and len(candidates) >= 5 and gra_before >= 7.5:
        issues.append("Multiple visible sentence-level problems were detected.")
        evidence.append("Detected issues: " + "; ".join([f"`{c.get('snippet', '')}`" for c in candidates[:3]]))
        moderate_points += 2
    elif len(candidates) >= 6 and gra_before >= 8.0:
        issues.append("There are several visible surface-level problems for a very high GRA band.")
        evidence.append("Detected issues: " + "; ".join([f"`{c.get('snippet', '')}`" for c in candidates[:3]]))
        moderate_points += 1

    if float(gra_feats.get("gf_lowercase_sent_start_ratio", 0.0)) > 0.14 and gra_before >= 7.5:
        issues.append("Sentence capitalization control looks inconsistent.")
        evidence.append(f"gf_lowercase_sent_start_ratio={gra_feats.get('gf_lowercase_sent_start_ratio', 0.0):.2f}")
        moderate_points += 1

    if float(gra_feats.get("gf_missing_terminal_punct", 0.0)) >= 1.0 and gra_before >= 8.0:
        issues.append("The essay ends without terminal punctuation.")
        evidence.append("gf_missing_terminal_punct=1")
        moderate_points += 1

    if float(gra_feats.get("gf_repeated_word_ratio", 0.0)) > 0.03 and gra_before >= 7.5:
        issues.append("Repeated-word patterns suggest weaker sentence control.")
        evidence.append(f"gf_repeated_word_ratio={gra_feats.get('gf_repeated_word_ratio', 0.0):.3f}")
        moderate_points += 1

    if float(gra_feats.get("gf_repeated_punct_ratio", 0.0)) > 0.03 and gra_before >= 8.0:
        issues.append("Punctuation control may be less steady than a very high GRA score suggests.")
        evidence.append(f"gf_repeated_punct_ratio={gra_feats.get('gf_repeated_punct_ratio', 0.0):.3f}")
        moderate_points += 1

    total_points = severe_points * 2 + moderate_points

    recommended_cap = gra_before
    risk_level = "low"

    if severe_points >= 2:
        recommended_cap = min(gra_before, 6.0)
        risk_level = "high"
    elif total_points >= 4 and gra_before >= 7.0:
        recommended_cap = min(gra_before, 6.5)
        risk_level = "high"
    elif total_points >= 3 and gra_before >= 7.0:
        recommended_cap = min(gra_before, 7.0)
        risk_level = "medium"
    elif total_points >= 2 and gra_before >= 7.5:
        recommended_cap = min(gra_before, 7.0)
        risk_level = "medium"
    elif len(candidates) >= 3 and gra_before >= 8.0:
        recommended_cap = min(gra_before, 7.5)
        risk_level = "medium"

    adjusted_scores = dict(pred_scores)
    if recommended_cap < gra_before:
        adjusted_scores = apply_band_cap(pred_scores, "GRA", recommended_cap)
        state["pred_scores"] = adjusted_scores

    result = {
        "criterion": "GRA",
        "checked": True,
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "recommended_cap": round_to_half_band(recommended_cap),
        "score_before": gra_before,
        "score_after": float((state.get("pred_scores") or pred_scores).get("GRA", gra_before)),
        "num_candidates": len(candidates),
    }
    state.setdefault("score_checks", {})["GRA"] = result

    return {
        "criterion": "GRA",
        "risk_level": risk_level,
        "issues": issues,
        "evidence": evidence,
        "num_candidates": len(candidates),
        "score_adjustment": {
            "GRA_before": gra_before,
            "GRA_after": float((state.get("pred_scores") or pred_scores).get("GRA", gra_before)),
            "Overall_after": float((state.get("pred_scores") or pred_scores).get("Overall", pred_scores.get("Overall", 0.0))),
        },
    }
def tool_check_high_band_consistency(prompt, essay, state):
    pred_scores = dict(state.get("pred_scores") or {})
    score_checks = dict(state.get("score_checks") or {})
    top_cases = list(state.get("top_cases") or [])

    result = {
        "checked": True,
        "used_top_cases": len(top_cases),
        "used_quality_retrieval": bool(QUALITY_INDEX),
        "decisions": {},
        "summary": [],
    }

    if not pred_scores or not score_checks:
        state["high_band_consistency"] = result
        return result

    task_info = state.get("task_check") or {}
    task_verdict = str(task_info.get("verdict", "")).lower()
    task_missing = task_info.get("missing_parts", []) or []
    task_risk = str(task_info.get("risk_level", "low")).lower()
    task_issues = task_info.get("issues", []) or []

    tr_feats = extract_tr_features(prompt, essay)
    cc_feats = extract_cc_features(prompt, essay)
    lr_feats = extract_lr_features(prompt, essay)
    gra_feats = extract_gra_features(prompt, essay)

    prompt_type = detect_prompt_type(prompt)
    paras = split_paragraphs(essay)
    words = [str(w).lower() for w in tokenize_words(essay)]
    content_words = [w for w in words if len(w) >= 4 and w not in STOPWORDS]
    freq = Counter(content_words)
    repeated_terms = [(w, c) for w, c in freq.items() if c >= 3]
    repeated_terms = sorted(repeated_terms, key=lambda x: (-x[1], x[0]))[:6]

    lower_essay = str(essay).lower()
    repeated_linkers = 0
    for linker in [
        "firstly", "secondly", "thirdly", "moreover", "besides",
        "however", "therefore", "in conclusion",
        "on the one hand", "on the other hand"
    ]:
        if lower_essay.count(linker) >= 2:
            repeated_linkers += 1

    candidates = extract_gra_error_candidates_v2(essay, max_candidates=10, max_per_type=4)
    severe_errors = [c for c in candidates if int(c.get("severity", 1)) >= 3]

    for crit in ["TR", "CC", "LR", "GRA"]:
        chk = score_checks.get(crit) or {}
        before = float(chk.get("score_before", pred_scores.get(crit, 0.0)))
        after = float(pred_scores.get(crit, before))
        risk_level = str(chk.get("risk_level", "low")).lower()

        decision = {
            "criterion": crit,
            "score_before": before,
            "score_after_before_restore": after,
            "risk_level": risk_level,
            "retrieved_support_count": 0,
            "retrieved_mean": None,
            "positive_signal_score": 0,
            "action": "keep",
            "restored_to": after,
            "reason": "",
        }

        if before < 7.0 or after >= before:
            decision["reason"] = "No high-band cap to review."
            result["decisions"][crit] = decision
            continue

        retrieved_scores = []

        quality_cases = retrieve_quality_cases_for_criterion(
            criterion=crit,
            prompt=prompt,
            essay=essay,
            state=state,
            store_name="main_quality",
            top_k_stage1=40,
            top_k_final=5,
            target_before=before,
            allowed_band_gap=0.5 if before >= 8.0 else 1.0,
        )

        if quality_cases:
            result["used_quality_retrieval"] = True
            for case in quality_cases:
                case_val = _quality_case_score(case, crit, None)
                if case_val is not None:
                    retrieved_scores.append(case_val)
        else:
            for case in top_cases:
                case_val = None
                for key in [crit, crit.lower(), crit.title()]:
                    if key in case and case[key] is not None and str(case[key]).strip() != "":
                        try:
                            case_val = float(case[key])
                            break
                        except Exception:
                            pass
                if case_val is not None:
                    retrieved_scores.append(case_val)

        if retrieved_scores:
            retrieved_mean = float(np.mean(retrieved_scores))
            support_count = sum(1 for x in retrieved_scores if x >= (before - 0.5))
            decision["retrieved_support_count"] = int(support_count)
            decision["retrieved_mean"] = round(retrieved_mean, 3)
        else:
            retrieved_mean = None
            support_count = 0

        positive_signal_score = 0

        if crit == "TR":
            coverage = float(tr_feats.get("tr_prompt_keyword_coverage", 0.0))
            sim = float(tr_feats.get("tr_prompt_essay_sim", 0.0))
            has_opinion = float(tr_feats.get("tr_has_opinion", 0.0)) >= 0.5
            has_both_views = float(tr_feats.get("tr_has_both_views", 0.0)) >= 0.5
            has_example = float(tr_feats.get("tr_has_example", 0.0)) >= 0.5
            has_conclusion = float(tr_feats.get("tr_has_conclusion", 0.0)) >= 0.5
            word_cnt = float(tr_feats.get("tr_word_count", 0.0))

            if coverage >= 0.12:
                positive_signal_score += 1
            if sim >= 0.34:
                positive_signal_score += 1
            if word_cnt >= 250:
                positive_signal_score += 1
            if has_example:
                positive_signal_score += 1
            if has_conclusion:
                positive_signal_score += 1

            if prompt_type == "agree_disagree":
                if has_opinion:
                    positive_signal_score += 1
            elif prompt_type in ["both_views", "both_views_opinion"]:
                if has_both_views:
                    positive_signal_score += 1
            else:
                positive_signal_score += 1

            if task_risk in ["medium", "high"] or len(task_issues) > 0:
                positive_signal_score -= 2

        elif crit == "CC":
            num_paragraphs = float(cc_feats.get("cc_num_paragraphs", len(paras)))
            avg_para_len = float(cc_feats.get("cc_avg_paragraph_len", 0.0))
            avg_sent_len = float(cc_feats.get("cc_avg_sentence_len", 0.0))
            sent_len_std = float(cc_feats.get("cc_sentence_len_std", 0.0))
            marker_count = float(cc_feats.get("cc_discourse_marker_count", 0.0))
            marker_div = float(cc_feats.get("cc_discourse_marker_diversity", 0.0))

            if num_paragraphs >= 4:
                positive_signal_score += 2
            elif num_paragraphs >= 3:
                positive_signal_score += 1
            if 45 <= avg_para_len <= 120:
                positive_signal_score += 1
            if 12 <= avg_sent_len <= 24:
                positive_signal_score += 1
            if sent_len_std <= 10:
                positive_signal_score += 1
            if marker_count >= 5:
                positive_signal_score += 1
            if marker_div >= 0.55:
                positive_signal_score += 1
            if repeated_linkers <= 1:
                positive_signal_score += 1

        elif crit == "LR":
            repetition_ratio = float(lr_feats.get("lr_repetition_ratio", 0.0))
            unique_ratio = float(lr_feats.get("lr_unique_word_ratio", 0.0))
            long_word_ratio = float(lr_feats.get("lr_long_word_ratio", 0.0))
            lexical_density = float(lr_feats.get("lr_lexical_density_proxy", 0.0))
            root_ttr = float(lr_feats.get("lr_root_ttr", 0.0))
            avg_word_len = float(lr_feats.get("lr_avg_word_len", 0.0))

            if repetition_ratio <= 0.09:
                positive_signal_score += 1
            if unique_ratio >= 0.46:
                positive_signal_score += 1
            if root_ttr >= 4.0:
                positive_signal_score += 1
            if long_word_ratio >= 0.17:
                positive_signal_score += 1
            if lexical_density >= 0.50:
                positive_signal_score += 1
            if avg_word_len >= 4.4:
                positive_signal_score += 1
            if len(repeated_terms) <= 2:
                positive_signal_score += 1

        elif crit == "GRA":
            lowercase_start = float(gra_feats.get("gf_lowercase_sent_start_ratio", 0.0))
            missing_terminal = float(gra_feats.get("gf_missing_terminal_punct", 0.0))
            repeated_word_ratio = float(gra_feats.get("gf_repeated_word_ratio", 0.0))
            repeated_punct_ratio = float(gra_feats.get("gf_repeated_punct_ratio", 0.0))
            lowercase_i_ratio = float(gra_feats.get("gf_lowercase_i_ratio", 0.0))
            long_sent_ratio = float(gra_feats.get("gf_long_sentence_ratio", 0.0))

            if len(severe_errors) == 0:
                positive_signal_score += 2
            if len(candidates) <= 2:
                positive_signal_score += 2
            elif len(candidates) <= 3:
                positive_signal_score += 1
            if lowercase_start <= 0.02:
                positive_signal_score += 1
            if missing_terminal == 0:
                positive_signal_score += 1
            if repeated_word_ratio <= 0.010:
                positive_signal_score += 1
            if repeated_punct_ratio <= 0.015:
                positive_signal_score += 1
            if lowercase_i_ratio == 0.0:
                positive_signal_score += 1
            if long_sent_ratio <= 0.28:
                positive_signal_score += 1

        decision["positive_signal_score"] = int(positive_signal_score)

        strong_support = (support_count >= 2 and positive_signal_score >= 6)
        moderate_support = (support_count >= 1 and positive_signal_score >= 5)

        if crit == "GRA" and positive_signal_score >= 8 and len(severe_errors) == 0 and support_count >= 1:
            moderate_support = True

        new_score = after
        reason = "Guardrail kept."
        action = "keep"

        if crit == "GRA":
            if len(severe_errors) >= 2:
                strong_support = False
                moderate_support = False
            elif len(severe_errors) == 1 and risk_level == "high":
                strong_support = False

        if crit == "TR" and (task_verdict in ["partial_mismatch", "off_topic"] or len(task_missing) > 0):
            strong_support = False
            moderate_support = False

        if strong_support:
            if risk_level == "medium":
                new_score = before
                action = "restore_full"
                reason = "Criterion-specific quality retrieval strongly supports the original high band."
            elif risk_level == "high":
                new_score = min(before, after + 0.5)
                action = "restore_half"
                reason = "Quality retrieval supports a partial restore, but the cap risk is still high."
        elif moderate_support:
            if risk_level in ["medium", "high"]:
                new_score = min(before, after + 0.5)
                action = "restore_half"
                reason = "Criterion-specific quality retrieval provides moderate support, so the cap is softened."
            else:
                action = "keep"
                reason = "Support exists, but no restore was needed."
        else:
            action = "keep"
            reason = "Not enough criterion-specific quality support to relax the cap."

        new_score = round_to_half_band(float(new_score))

        if new_score > after:
            pred_scores[crit] = new_score

        decision["action"] = action
        decision["restored_to"] = float(pred_scores.get(crit, after))
        decision["reason"] = reason
        result["decisions"][crit] = decision

        if action != "keep":
            result["summary"].append(f"{crit}: {after:.1f} -> {pred_scores[crit]:.1f} ({action})")

    for crit in ["TR", "CC", "LR", "GRA"]:
        pred_scores[crit] = round_to_half_band(float(pred_scores.get(crit, 0.0)))

    pred_scores["Overall"] = round_to_half_band(np.mean([
        pred_scores["TR"],
        pred_scores["CC"],
        pred_scores["LR"],
        pred_scores["GRA"],
    ]))
    state["pred_scores"] = pred_scores
    result["overall_after"] = float(pred_scores["Overall"])

    state["high_band_consistency"] = result
    return result

    task_info = state.get("task_check") or {}
    task_verdict = str(task_info.get("verdict", "")).lower()
    task_missing = task_info.get("missing_parts", []) or []

    # thêm 2 dòng này để tránh NameError
    task_risk = str(task_info.get("risk_level", "low")).lower()
    task_issues = task_info.get("issues", []) or []

    tr_feats = extract_tr_features(prompt, essay)
    cc_feats = extract_cc_features(prompt, essay)
    lr_feats = extract_lr_features(prompt, essay)
    gra_feats = extract_gra_features(prompt, essay)

    prompt_type = detect_prompt_type(prompt)
    paras = split_paragraphs(essay)
    words = [str(w).lower() for w in tokenize_words(essay)]
    content_words = [w for w in words if len(w) >= 4 and w not in STOPWORDS]
    freq = Counter(content_words)
    repeated_terms = [(w, c) for w, c in freq.items() if c >= 3]
    repeated_terms = sorted(repeated_terms, key=lambda x: (-x[1], x[0]))[:6]

    lower_essay = str(essay).lower()
    repeated_linkers = 0
    for linker in [
        "firstly", "secondly", "thirdly", "moreover", "besides",
        "however", "therefore", "in conclusion",
        "on the one hand", "on the other hand"
    ]:
        if lower_essay.count(linker) >= 2:
            repeated_linkers += 1

    candidates = extract_gra_error_candidates_v2(essay, max_candidates=10, max_per_type=4)
    severe_errors = [c for c in candidates if int(c.get("severity", 1)) >= 3]

    for crit in ["TR", "CC", "LR", "GRA"]:
        chk = score_checks.get(crit) or {}
        before = float(chk.get("score_before", pred_scores.get(crit, 0.0)))
        after = float(pred_scores.get(crit, before))
        risk_level = str(chk.get("risk_level", "low")).lower()
        issues = chk.get("issues", []) or []

        decision = {
            "criterion": crit,
            "score_before": before,
            "score_after_before_restore": after,
            "risk_level": risk_level,
            "retrieved_support_count": 0,
            "retrieved_mean": None,
            "positive_signal_score": 0,
            "action": "keep",
            "restored_to": after,
            "reason": "",
        }

        if before < 7.0 or after >= before:
            decision["reason"] = "No high-band cap to review."
            result["decisions"][crit] = decision
            continue

        retrieved_scores = []
        for case in top_cases:
            case_val = None
            for key in [crit, crit.lower(), crit.title()]:
                if key in case and case[key] is not None and str(case[key]).strip() != "":
                    try:
                        case_val = float(case[key])
                        break
                    except:
                        pass
            if case_val is not None:
                retrieved_scores.append(case_val)

        if retrieved_scores:
            retrieved_mean = float(np.mean(retrieved_scores))
            support_count = sum(1 for x in retrieved_scores if x >= (before - 0.5))
            decision["retrieved_support_count"] = int(support_count)
            decision["retrieved_mean"] = round(retrieved_mean, 3)
        else:
            retrieved_mean = None
            support_count = 0

        positive_signal_score = 0

        if crit == "TR":
            coverage = float(tr_feats.get("tr_prompt_keyword_coverage", 0.0))
            sim = float(tr_feats.get("tr_prompt_essay_sim", 0.0))
            has_opinion = float(tr_feats.get("tr_has_opinion", 0.0)) >= 0.5
            has_both_views = float(tr_feats.get("tr_has_both_views", 0.0)) >= 0.5
            has_example = float(tr_feats.get("tr_has_example", 0.0)) >= 0.5
            has_conclusion = float(tr_feats.get("tr_has_conclusion", 0.0)) >= 0.5
            word_cnt = float(tr_feats.get("tr_word_count", 0.0))

            if coverage >= 0.12:
                positive_signal_score += 1
            if sim >= 0.34:
                positive_signal_score += 1
            if word_cnt >= 250:
                positive_signal_score += 1
            if has_example:
                positive_signal_score += 1
            if has_conclusion:
                positive_signal_score += 1

            if prompt_type == "agree_disagree":
                if has_opinion:
                    positive_signal_score += 1
            elif prompt_type in ["both_views", "both_views_opinion"]:
                if has_both_views:
                    positive_signal_score += 1
            else:
                positive_signal_score += 1

            if task_risk in ["medium", "high"] or len(task_issues) > 0:
                positive_signal_score -= 2

        elif crit == "CC":
            num_paragraphs = float(cc_feats.get("cc_num_paragraphs", len(paras)))
            avg_para_len = float(cc_feats.get("cc_avg_paragraph_len", 0.0))
            avg_sent_len = float(cc_feats.get("cc_avg_sentence_len", 0.0))
            sent_len_std = float(cc_feats.get("cc_sentence_len_std", 0.0))
            marker_count = float(cc_feats.get("cc_discourse_marker_count", 0.0))
            marker_div = float(cc_feats.get("cc_discourse_marker_diversity", 0.0))

            if num_paragraphs >= 4:
                positive_signal_score += 2
            elif num_paragraphs >= 3:
                positive_signal_score += 1

            if 45 <= avg_para_len <= 120:
                positive_signal_score += 1
            if 12 <= avg_sent_len <= 24:
                positive_signal_score += 1
            if sent_len_std <= 10:
                positive_signal_score += 1
            if marker_count >= 5:
                positive_signal_score += 1
            if marker_div >= 0.55:
                positive_signal_score += 1
            if repeated_linkers <= 1:
                positive_signal_score += 1

        elif crit == "LR":
            repetition_ratio = float(lr_feats.get("lr_repetition_ratio", 0.0))
            unique_ratio = float(lr_feats.get("lr_unique_word_ratio", 0.0))
            long_word_ratio = float(lr_feats.get("lr_long_word_ratio", 0.0))
            lexical_density = float(lr_feats.get("lr_lexical_density_proxy", 0.0))
            root_ttr = float(lr_feats.get("lr_root_ttr", 0.0))
            avg_word_len = float(lr_feats.get("lr_avg_word_len", 0.0))

            if repetition_ratio <= 0.09:
                positive_signal_score += 1
            if unique_ratio >= 0.46:
                positive_signal_score += 1
            if root_ttr >= 4.0:
                positive_signal_score += 1
            if long_word_ratio >= 0.17:
                positive_signal_score += 1
            if lexical_density >= 0.50:
                positive_signal_score += 1
            if avg_word_len >= 4.4:
                positive_signal_score += 1
            if len(repeated_terms) <= 2:
                positive_signal_score += 1

        elif crit == "GRA":
            lowercase_start = float(gra_feats.get("gf_lowercase_sent_start_ratio", 0.0))
            missing_terminal = float(gra_feats.get("gf_missing_terminal_punct", 0.0))
            repeated_word_ratio = float(gra_feats.get("gf_repeated_word_ratio", 0.0))
            repeated_punct_ratio = float(gra_feats.get("gf_repeated_punct_ratio", 0.0))
            lowercase_i_ratio = float(gra_feats.get("gf_lowercase_i_ratio", 0.0))
            long_sent_ratio = float(gra_feats.get("gf_long_sentence_ratio", 0.0))

            if len(severe_errors) == 0:
                positive_signal_score += 2
            if len(candidates) <= 2:
                positive_signal_score += 2
            elif len(candidates) <= 3:
                positive_signal_score += 1
            if lowercase_start <= 0.02:
                positive_signal_score += 1
            if missing_terminal == 0:
                positive_signal_score += 1
            if repeated_word_ratio <= 0.010:
                positive_signal_score += 1
            if repeated_punct_ratio <= 0.015:
                positive_signal_score += 1
            if lowercase_i_ratio == 0.0:
                positive_signal_score += 1
            if long_sent_ratio <= 0.28:
                positive_signal_score += 1

        decision["positive_signal_score"] = int(positive_signal_score)

        strong_support = (support_count >= 2 and positive_signal_score >= 6)
        moderate_support = (support_count >= 2 and positive_signal_score >= 4)

        new_score = after
        reason = "Guardrail kept."
        action = "keep"

        if crit == "GRA":
            if len(severe_errors) >= 2:
                strong_support = False
                moderate_support = False
            elif len(severe_errors) == 1 and risk_level == "high":
                strong_support = False

        if crit == "TR" and (task_verdict in ["partial_mismatch", "off_topic"] or len(task_missing) > 0):
            strong_support = False
            moderate_support = False

        if strong_support:
            if risk_level == "medium":
                new_score = before
                action = "restore_full"
                reason = "Retrieved anchor essays support a high band and current essay has strong positive signals."
            elif risk_level == "high":
                new_score = min(before, after + 0.5)
                action = "restore_half"
                reason = "There is high-band support from retrieval, but existing risk is still too strong for a full restore."
        elif moderate_support:
            if risk_level == "medium":
                new_score = min(before, after + 0.5)
                action = "restore_half"
                reason = "Retrieved anchor essays moderately support a higher band, so the cap is softened."
            else:
                action = "keep"
                reason = "Support exists, but the detected risk is still too strong."
        else:
            action = "keep"
            reason = "Not enough retrieved support or positive evidence to relax the cap."

        new_score = round_to_half_band(float(new_score))

        if new_score > after:
            pred_scores[crit] = new_score

        decision["action"] = action
        decision["restored_to"] = float(pred_scores.get(crit, after))
        decision["reason"] = reason
        result["decisions"][crit] = decision

        if action != "keep":
            result["summary"].append(f"{crit}: {after:.1f} -> {pred_scores[crit]:.1f} ({action})")

    for crit in ["TR", "CC", "LR", "GRA"]:
        pred_scores[crit] = round_to_half_band(float(pred_scores.get(crit, 0.0)))

    pred_scores["Overall"] = round_to_half_band(np.mean([
        pred_scores["TR"],
        pred_scores["CC"],
        pred_scores["LR"],
        pred_scores["GRA"],
    ]))
    state["pred_scores"] = pred_scores
    result["overall_after"] = float(pred_scores["Overall"])

    state["high_band_consistency"] = result
    return result


In [17]:
def tool_retrieve_similar_essays(prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    res = retrieve_similar_essays_for_inference(prompt, essay, df, db_embeddings, top_k_vector, top_k_final, pred_scores=state.get("pred_scores"))
    state["top_cases"] = res["top_cases"]
    return {"status": f"Retrieved {len(res['top_cases'])} cases"}

Tool Generate Feedback


In [18]:
def limitation_mentions_structure_problem(text):
    text = str(text or "").lower()
    patterns = [
        "clear introduction, body, and conclusion",
        "introduction, body, and conclusion",
        "lacks proper paragraphing",
        "better paragraphing",
        "clearer paragraphs",
        "organize the essay into clear paragraphs",
        "breaking the essay into separate paragraphs",
        "separate paragraphs for the introduction, body, and conclusion",
    ]
    return any(p in text for p in patterns)


def essay_has_basic_structure(prompt, essay):
    paras = split_paragraphs(essay)
    num_paras = len(paras)
    has_conc = has_conclusion(essay)

    intro_like = False
    if num_paras >= 1:
        first_para = normalize_text(paras[0])
        intro_markers = [
            "i believe", "i think", "in my opinion", "personally",
            "this essay", "i agree", "i disagree", "i truly believe"
        ]
        intro_like = any(m in first_para for m in intro_markers)

    body_like = num_paras >= 2
    conclusion_like = has_conc or num_paras >= 3

    return {
        "num_paragraphs": num_paras,
        "has_intro_like": intro_like,
        "has_body_like": body_like,
        "has_conclusion_like": conclusion_like,
        "has_basic_3part_structure": (num_paras >= 3) or (intro_like and body_like and conclusion_like),
    }



def contains_any(text, patterns):
    text = str(text or "").lower()
    return any(p.lower() in text for p in patterns)


def get_band_aligned_strength(criterion, band):
    band = float(band)

    if criterion == "TR":
        if band <= 5.0:
            return "The essay presents a clear position on the topic and includes some relevant supporting ideas."
        elif band <= 6.0:
            return "The essay addresses the topic clearly and supports the main opinion with generally relevant reasons."
        else:
            return "The essay presents a clear and well-supported position throughout the response."

    if criterion == "CC":
        if band <= 5.0:
            return "The essay shows a basic overall structure with an introduction, body, and conclusion, although progression is not always clear."
        elif band <= 6.0:
            return "The essay is generally organized logically, with a clear overall progression of ideas."
        else:
            return "The essay is well organized and ideas progress clearly across the response."

    if criterion == "LR":
        if band <= 5.0:
            return "The essay uses some topic-related vocabulary about tax, government, and public support, although word choice is often inaccurate or awkward."
        elif band <= 6.0:
            return "The essay uses a sufficient range of vocabulary to discuss the topic, though some word choices are imprecise."
        else:
            return "The essay uses a varied and generally precise range of vocabulary to express ideas."

    if criterion == "GRA":
        if band <= 5.0:
            return "The essay attempts to form complete sentences, but grammar and sentence control are inconsistent."
        elif band <= 6.0:
            return "The essay shows some control of grammar and sentence structure, although errors remain noticeable."
        else:
            return "The essay demonstrates generally accurate grammar and effective control of sentence structure."

    return ""


def rewrite_tr_limitation_if_needed(prompt, essay, pred_scores, tr_block):
    tr_block = dict(tr_block or {})
    limitation = str(tr_block.get("Limitation", "")).strip()
    revision = str(tr_block.get("Revision", "")).strip()

    prompt_type = detect_prompt_type(prompt)
    structure_info = essay_has_basic_structure(prompt, essay)
    has_example_flag = has_example(essay)

    # 1) Agree/Disagree mà lại đòi opposing view -> sửa
    if prompt_type == "agree_disagree" and contains_any(
        limitation,
        [
            "opposing viewpoint",
            "opposing viewpoints",
            "opposite view",
            "counterargument",
            "both sides",
            "other side",
        ],
    ):
        tr_block["Limitation"] = (
            "The writer’s position is clear, but the supporting reasons are developed only briefly and the examples are not specific enough to make the argument fully convincing."
        )
        tr_block["Revision"] = (
            "Develop each supporting reason more clearly and add one specific example showing how tax money benefits society."
        )
        return tr_block

    # 2) Bài đã có basic structure mà TR vẫn chê thiếu intro/body/conclusion -> sửa
    if structure_info["has_basic_3part_structure"] and limitation_mentions_structure_problem(limitation):
        tr_block["Limitation"] = (
            "The response stays on topic and has a clear overall position, but the main ideas are explained only briefly and supporting examples remain too general."
        )
        tr_block["Revision"] = (
            "Keep the current essay structure, but explain each reason in more detail and add a clearer real-world example."
        )
        return tr_block

    # 3) Band thấp + chưa có ví dụ rõ mà limitation đòi personal knowledge quá cứng -> làm mềm
    if float(pred_scores["TR"]) <= 5.0 and not has_example_flag and contains_any(
        limitation,
        [
            "specific examples from personal experience",
            "personal knowledge or experience",
            "own knowledge or experience",
        ],
    ):
        tr_block["Limitation"] = (
            "The ideas are relevant to the topic, but they are developed quite generally and need clearer support."
        )
        tr_block["Revision"] = (
            "Add one simple, specific example to support the main argument more clearly."
        )

    return tr_block


def rewrite_cc_limitation_if_needed(prompt, essay, pred_scores, cc_block):
    cc_block = dict(cc_block or {})
    limitation = str(cc_block.get("Limitation", "")).strip()

    structure_info = essay_has_basic_structure(prompt, essay)
    cc_feat = extract_cc_features(prompt, essay)
    num_paras = float(cc_feat.get("cc_num_paragraphs", 0))

    # Bài đã chia đoạn mà vẫn bị chê lacks proper paragraphing -> sửa
    if num_paras >= 3 and limitation_mentions_structure_problem(limitation):
        cc_block["Limitation"] = (
            "The essay has a basic paragraph structure, but the progression between ideas is not always smooth and some sentences are difficult to follow."
        )
        cc_block["Revision"] = (
            "Keep the current paragraph structure, but make each paragraph focus on one clear point and use simpler transitions between ideas."
        )
        return cc_block

    # Có cấu trúc cơ bản nhưng flow chưa tốt -> dùng phrasing mềm hơn
    if structure_info["has_basic_3part_structure"] and contains_any(
        limitation,
        ["clear introduction, body, and conclusion", "better paragraphing"]
    ):
        cc_block["Limitation"] = (
            "The overall structure is visible, but cohesion is weakened because some ideas connect abruptly and sentence flow is not always clear."
        )
        cc_block["Revision"] = (
            "Link ideas more smoothly within and between paragraphs, and avoid abrupt jumps from one point to another."
        )

    return cc_block


def apply_feedback_consistency_rules(feedback, prompt, essay, pred_scores):
    feedback = normalize_feedback_dict(feedback)

    # Strength theo band
    for crit in ["TR", "CC", "LR", "GRA"]:
        feedback[crit]["Strength"] = get_band_aligned_strength(crit, pred_scores[crit])

    # Sửa TR / CC bị chê sai kiểu
    feedback["TR"] = rewrite_tr_limitation_if_needed(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        tr_block=feedback.get("TR", {}),
    )

    feedback["CC"] = rewrite_cc_limitation_if_needed(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        cc_block=feedback.get("CC", {}),
    )

    # GRA có nhiều lỗi nhìn thấy được thì strength phải bảo thủ
    gra_candidates = extract_gra_error_candidates_v2(
        essay,
        max_candidates=6,
        max_per_type=2,
    )
    if len(gra_candidates) >= 2:
        feedback["GRA"]["Strength"] = get_band_aligned_strength(
            "GRA",
            min(float(pred_scores["GRA"]), 5.5)
        )

    # LR limitation đã nói awkward/repetition thì strength không được khen quá tay
    lr_lim = str(feedback.get("LR", {}).get("Limitation", ""))
    if contains_any(
        lr_lim,
        [
            "awkward phrasing",
            "repetitive wording",
            "imprecise",
            "awkward wording",
            "repetition",
        ],
    ):
        feedback["LR"]["Strength"] = get_band_aligned_strength(
            "LR",
            min(float(pred_scores["LR"]), 5.5)
        )

    return feedback

def count_backtick_spans(text):
    return len(re.findall(r"`[^`]+`", str(text or "")))


def build_gra_limitation_from_candidates(candidates):
    candidates = candidates or []
    if not candidates:
        return ""

    parts = []
    for c in candidates[:3]:
        parts.append(f"`{c['snippet']}` shows that {c['reason']}")

    if len(parts) == 1:
        body = parts[0]
    elif len(parts) == 2:
        body = parts[0] + "; also, " + parts[1]
    else:
        body = parts[0] + "; " + parts[1] + "; and " + parts[2]

    return f"For example, {body}. These errors reduce sentence-level accuracy and clarity."


def strengthen_gra_limitation(feedback, essay, min_anchors=2):
    if not isinstance(feedback, dict):
        return feedback

    feedback = {
        k: (v.copy() if isinstance(v, dict) else v)
        for k, v in feedback.items()
    }

    gra_block = feedback.get("GRA", {}) or {}
    limitation = str(gra_block.get("Limitation", "")).strip()

    candidates = extract_gra_error_candidates_v2(
        essay,
        max_candidates=6,
        max_per_type=2,
    )

    if len(candidates) >= min_anchors and count_backtick_spans(limitation) < min_anchors:
        rebuilt = build_gra_limitation_from_candidates(candidates[:3])
        if rebuilt:
            gra_block["Limitation"] = rebuilt
    elif not limitation and candidates:
        gra_block["Limitation"] = build_gra_limitation_from_candidates(candidates[:3])

    feedback["GRA"] = gra_block
    return feedback


def _clean_gra_snippet(text, max_len=180):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:max_len].rstrip(" ,;:")


def extract_gra_error_candidates_v2(essay, max_candidates=8, max_per_type=2):
    text = str(essay or "").strip()
    if not text:
        return []

    seen = set()
    candidates = []
    type_counter = Counter()

    def add(snippet, reason, category, severity=1):
        snippet = _clean_gra_snippet(snippet)
        if not snippet:
            return

        key = (snippet.lower(), reason.lower(), category.lower())
        if key in seen:
            return
        if type_counter[category] >= max_per_type:
            return

        seen.add(key)
        type_counter[category] += 1
        candidates.append({
            "snippet": snippet,
            "reason": reason,
            "category": category,
            "severity": severity,
        })

    sentences = re.split(r'(?<=[.!?])\s+|\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    for s in sentences:
        # 1) lowercase sentence start
        if re.match(r"^[a-z]", s):
            add(
                s,
                "the sentence starts with a lowercase letter",
                "capitalization",
                severity=2,
            )

        # 2) lowercase i
        if re.search(r"\bi\b", s):
            add(
                s,
                'the pronoun `I` is not capitalized',
                "capitalization",
                severity=2,
            )

        # 3) repeated punctuation
        if re.search(r"([,.!?])\1{1,}", s):
            add(
                s,
                "punctuation is repeated unnecessarily",
                "punctuation",
                severity=2,
            )

        # 4) repeated word
        if re.search(r"\b([A-Za-z]+)\s+\1\b", s, flags=re.I):
            add(
                s,
                "a word is repeated unnecessarily",
                "word_form",
                severity=2,
            )

        # 5) common subject-verb agreement
        if re.search(r"\b(people|students|children|parents|governments)\s+is\b", s, flags=re.I):
            add(
                s,
                "subject-verb agreement is inaccurate",
                "agreement",
                severity=3,
            )

        if re.search(r"\b(he|she|it)\s+are\b", s, flags=re.I):
            add(
                s,
                "subject-verb agreement is inaccurate",
                "agreement",
                severity=3,
            )

        if re.search(r"\b(they|we|you)\s+is\b", s, flags=re.I):
            add(
                s,
                "subject-verb agreement is inaccurate",
                "agreement",
                severity=3,
            )

        # 6) article a/an
        if re.search(r"\ba\s+[aeiouAEIOU][A-Za-z]+\b", s):
            add(
                s,
                "the article choice may be inaccurate (`a/an` mismatch)",
                "article",
                severity=2,
            )

        if re.search(r"\ban\s+[^aeiouAEIOU\W][A-Za-z]+\b", s):
            add(
                s,
                "the article choice may be inaccurate (`a/an` mismatch)",
                "article",
                severity=2,
            )

        # 7) comparative form
        if re.search(r"\b(more|most)\s+(better|worse|easier|harder|faster|slower)\b", s, flags=re.I):
            add(
                s,
                "the comparative form is inaccurate",
                "comparative",
                severity=2,
            )

        # 8) modal + wrong finite verb
        if re.search(r"\b(can|could|should|must|may|might|will|would)\s+\w+(s|ed)\b", s, flags=re.I):
            add(
                s,
                "the verb form after a modal may be inaccurate",
                "verb_form",
                severity=2,
            )

        # 9) run-on heuristic
        comma_count = s.count(",")
        word_len = len(re.findall(r"\b[A-Za-z']+\b", s))
        if word_len >= 32 and comma_count >= 2 and not re.search(r"[;:]", s):
            add(
                s,
                "the sentence is likely too long and may be a run-on structure",
                "sentence_boundary",
                severity=3,
            )

    # 10) missing final punctuation
    if text and text[-1] not in ".!?":
        add(
            text[-120:],
            "the essay ends without terminal punctuation",
            "punctuation",
            severity=1,
        )

    # ưu tiên lỗi nặng hơn và snippet ngắn vừa phải
    candidates = sorted(
        candidates,
        key=lambda x: (-x["severity"], len(x["snippet"]))
    )

    return candidates[:max_candidates]


def gra_limitation_has_anchor(text):
    text = str(text or "").strip().lower()
    if not text:
        return False

    if "`" in text:
        return True

    anchor_markers = [
        "for example",
        "for instance",
        "in the phrase",
        "in the sentence",
        "the phrase",
        "the sentence",
    ]
    return any(x in text for x in anchor_markers)


def force_gra_limitation_anchor(feedback, essay):
    if not isinstance(feedback, dict):
        return feedback

    feedback = {
        k: (v.copy() if isinstance(v, dict) else v)
        for k, v in feedback.items()
    }

    gra_block = feedback.get("GRA", {})
    if not isinstance(gra_block, dict):
        gra_block = {}

    limitation = str(gra_block.get("Limitation", "")).strip()

    if gra_limitation_has_anchor(limitation):
        feedback["GRA"] = gra_block
        return feedback

    candidates = extract_gra_error_candidates_v2(essay, max_candidates=1, max_per_type=1)
    if not candidates:
        feedback["GRA"] = gra_block
        return feedback

    c = candidates[0]
    prefix = f'For example, `{c["snippet"]}` shows that {c["reason"]}.'

    if limitation:
        gra_block["Limitation"] = prefix + " " + limitation
    else:
        gra_block["Limitation"] = prefix

    feedback["GRA"] = gra_block
    return feedback


def format_gra_candidates_for_prompt(candidates, max_items=5):
    candidates = candidates or []
    if not candidates:
        return "[]"

    lines = []
    for i, c in enumerate(candidates[:max_items], start=1):
        lines.append(
            f"{i}. category={c['category']} | severity={c['severity']} | "
            f"evidence=`{c['snippet']}` | issue={c['reason']}"
        )
    return "\n".join(lines)

def build_generate_feedback_prompt(prompt, essay, pred_scores, top_cases):
    refs_text = format_top_cases_for_prompt(top_cases, max_cases=3, max_chars_each=900)

    return f"""
You are a strict IELTS Writing Task 2 feedback writer.

Your task:
Write criterion-level feedback for the given essay using the FIXED predicted scores below.
The feedback must be faithful to the essay itself. The retrieved reference cases are only grounding support, not text to copy.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Retrieved reference cases:
{refs_text}

Hard rules:
1. NEVER change the predicted scores.
2. Do NOT mention any alternative band score.
3. Do NOT invent weaknesses that are not visible in the essay.
4. Each criterion must stay in its own scope:
   - TR: task response, relevance, coverage, position, development
   - CC: organization, progression, paragraphing, linking
   - LR: vocabulary range, precision, repetition, collocation, word choice
   - GRA: grammar, sentence structure, articles, prepositions, agreement, verb forms, punctuation
5. Do NOT mix criteria:
   - CC must not talk about grammar or vocabulary quality
   - LR must not talk about grammar or coherence
   - GRA must not talk about idea development or vocabulary range
6. For GRA, the Limitation field MUST begin with one concrete error, phrase, or sentence fragment copied from the essay.
7. IMPORTANT: the output must be valid JSON. Therefore, do NOT use raw double quotes inside JSON values.
8. For copied essay evidence, use backticks instead of double quotes.
9. Use this exact preferred pattern for GRA Limitation:
   For example, `...` shows ...
10. Generic GRA limitation such as "there are some grammatical errors" is not acceptable unless it starts with a copied example from the essay.
11. Keep the tone examiner-like, concise, and specific.
12. Each criterion must have exactly these three fields:
   - Strength
   - Limitation
   - Revision
13. Revision must be actionable and aligned with the criterion.
14. Do not copy wording from the retrieved reference essays.
15. Output valid JSON only. No explanation before or after the JSON block.

Output valid JSON only, in exactly this format:
{{
  "TR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "CC": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "LR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "GRA": {{
    "Strength": "...",
    "Limitation": "For example, `...` shows ...",
    "Revision": "..."
  }}
}}
""".strip()


def normalize_feedback_dict(obj):
    result = {}
    for crit in ["TR", "CC", "LR", "GRA"]:
        block = obj.get(crit, {}) if isinstance(obj, dict) else {}
        if not isinstance(block, dict):
            block = {}
        result[crit] = {
            "Strength": str(block.get("Strength", "")).strip(),
            "Limitation": str(block.get("Limitation", "")).strip(),
            "Revision": str(block.get("Revision", "")).strip(),
        }

        if crit == "GRA" and result[crit]["Limitation"]:
            result[crit]["Limitation"] = result[crit]["Limitation"].replace('\\"', '"')

    return result


def parse_feedback_block_text(block_text):
    if not isinstance(block_text, str):
        block_text = str(block_text)

    strength = ""
    limitation = ""
    revision = ""

    m1 = re.search(
        r"Strength:\s*(.*?)(?=\n\s*Limitation:|\Z)",
        block_text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    m2 = re.search(
        r"Limitation:\s*(.*?)(?=\n\s*Revision:|\Z)",
        block_text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    m3 = re.search(
        r"Revision:\s*(.*)",
        block_text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if m1:
        strength = m1.group(1).strip()
    if m2:
        limitation = m2.group(1).strip()
    if m3:
        revision = m3.group(1).strip()

    return {
        "Strength": strength,
        "Limitation": limitation,
        "Revision": revision,
    }


def parse_feedback_output(text):
    result = {
        "TR": {"Strength": "", "Limitation": "", "Revision": ""},
        "CC": {"Strength": "", "Limitation": "", "Revision": ""},
        "LR": {"Strength": "", "Limitation": "", "Revision": ""},
        "GRA": {"Strength": "", "Limitation": "", "Revision": ""},
    }

    if not isinstance(text, str):
        return result

    text = text.strip()

    parsed = safe_json_loads(text, default=None)
    if isinstance(parsed, dict):
        out = {}
        ok = False

        for c in ["TR", "CC", "LR", "GRA"]:
            block = parsed.get(c, {})
            if isinstance(block, dict):
                out[c] = {
                    "Strength": str(block.get("Strength", "")).strip(),
                    "Limitation": str(block.get("Limitation", "")).strip(),
                    "Revision": str(block.get("Revision", "")).strip(),
                }
                if any(out[c].values()):
                    ok = True
            elif isinstance(block, str):
                out[c] = parse_feedback_block_text(block)
                if any(out[c].values()):
                    ok = True
            else:
                out[c] = {
                    "Strength": "",
                    "Limitation": "",
                    "Revision": "",
                }

        if ok:
            return out

    text = text.replace("\r", "\n")
    patterns = {
        "TR": r"TR\s*:\s*(.*?)(?=\n(?:CC|LR|GRA)\s*:|\Z)",
        "CC": r"CC\s*:\s*(.*?)(?=\n(?:TR|LR|GRA)\s*:|\Z)",
        "LR": r"LR\s*:\s*(.*?)(?=\n(?:TR|CC|GRA)\s*:|\Z)",
        "GRA": r"GRA\s*:\s*(.*?)(?=\n(?:TR|CC|LR)\s*:|\Z)",
    }

    out = {}
    for crit, pat in patterns.items():
        m = re.search(pat, text, flags=re.S | re.I)
        block_text = m.group(1).strip() if m else ""
        out[crit] = parse_feedback_block_text(block_text)

    return out


def dedup_lr_gra_overlap(feedback):
    if not isinstance(feedback, dict):
        return feedback

    feedback = {
        k: (v.copy() if isinstance(v, dict) else v)
        for k, v in feedback.items()
    }

    lr_lim = str(feedback.get("LR", {}).get("Limitation", ""))
    gra_lim = str(feedback.get("GRA", {}).get("Limitation", ""))

    lr_snips = set(re.findall(r"`([^`]+)`", lr_lim))
    gra_snips = set(re.findall(r"`([^`]+)`", gra_lim))
    overlap = lr_snips & gra_snips

    if overlap:
        overlap_text = next(iter(overlap))
        feedback["LR"]["Limitation"] = (
            f"The essay contains some imprecise or awkward wording, and phrases such as `{overlap_text}` "
            f"reduce lexical naturalness."
        )
        feedback["LR"]["Revision"] = (
            "Replace awkward or repetitive wording with more natural collocations and avoid reusing the same imprecise phrase."
        )

    return feedback


def tool_generate_feedback(prompt, essay, state):
    pred_scores = state["pred_scores"]
    top_cases = state.get("top_cases", []) or []

    gra_candidates = extract_gra_error_candidates_v2(
        essay,
        max_candidates=6,
        max_per_type=2,
    )
    state["gra_candidates"] = gra_candidates

    refs_text = format_top_cases_for_prompt(
        top_cases=top_cases,
        prompt=prompt,
        max_cases=3,
        max_chars_each=900,
    )
    gra_candidates_text = format_gra_candidates_for_prompt(
        gra_candidates,
        max_items=5,
    )

    gen_prompt = f"""
You are a strict IELTS Writing Task 2 feedback writer.

Your task:
Write criterion-level feedback for the given essay using the FIXED predicted scores below.
The feedback must be faithful to the essay itself. The retrieved reference cases are only grounding support, not text to copy.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Retrieved reference cases:
{refs_text}

Pre-extracted GRA evidence candidates:
{gra_candidates_text}

Hard rules:
1. NEVER change the predicted scores.
2. Do NOT mention any alternative band score.
3. Do NOT invent weaknesses that are not visible in the essay.
4. Each criterion must stay in its own scope:
   - TR: task response, relevance, coverage, position, development
   - CC: organization, progression, paragraphing, linking
   - LR: vocabulary range, precision, repetition, collocation, word choice
   - GRA: grammar, sentence structure, articles, prepositions, agreement, verb forms, punctuation
5. Do NOT mix criteria.
6. LR limitation should mention at least one concrete word or phrase if it claims awkward or repetitive wording.
7. For GRA, the Limitation field MUST begin with concrete copied evidence from the essay.
8. Use backticks for copied evidence, not raw double quotes.
9. If 2 or more GRA evidence candidates are available, mention at least 2 concrete errors.
10. Keep the tone concise, examiner-like, and specific.
11. Strength must be realistic for the fixed score. For low scores, do not use phrases like `precise language`, `correct grammar`, or `no major errors`.
12. Each criterion must have exactly these three fields:
   - Strength
   - Limitation
   - Revision
13. Output valid JSON only.
14. Do not claim that the essay lacks introduction, body, and conclusion if these parts are already visibly present.
15. Do not claim that the essay lacks paragraphing if it already contains multiple paragraphs; instead describe unclear progression or weak cohesion.

Output valid JSON only, in exactly this format:
{{
  "TR": {{"Strength": "...", "Limitation": "...", "Revision": "..."}},
  "CC": {{"Strength": "...", "Limitation": "...", "Revision": "..."}},
  "LR": {{"Strength": "...", "Limitation": "...", "Revision": "..."}},
  "GRA": {{"Strength": "...", "Limitation": "For example, `...` shows ...", "Revision": "..."}}
}}
""".strip()

    raw = mistral_explain_writer(
        gen_prompt,
        temperature=0.0,
        max_new_tokens=900,
    )

    feedback = parse_feedback_output(raw)
    feedback = normalize_feedback_dict(feedback)
    feedback = strengthen_gra_limitation(
        feedback=feedback,
        essay=essay,
        min_anchors=2,
    )
    feedback = dedup_lr_gra_overlap(feedback)

    gra_block = feedback.get("GRA", {}) if isinstance(feedback, dict) else {}
    current_gra_limitation = str(gra_block.get("Limitation", "")).strip()
    if not current_gra_limitation and gra_candidates:
        gra_block["Limitation"] = build_gra_limitation_from_candidates(gra_candidates[:3])
        feedback["GRA"] = gra_block

    # NEW: deterministic consistency fix
    feedback = apply_feedback_consistency_rules(
        feedback=feedback,
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
    )

    state["feedback"] = feedback
    state["feedback_raw"] = str(raw)

    return {
        "feedback_generated": True,
        "criteria": list(feedback.keys()),
        "gra_candidate_count": len(gra_candidates),
        "gra_candidates_preview": gra_candidates[:3],
        "raw_preview": str(raw)[:800],
    }

Tool VerifyFeedBack

In [19]:
def extract_quoted_phrases(text):
    text = str(text or "")
    found = []

    for m in re.findall(r"`([^`]+)`", text):
        if m.strip():
            found.append(m.strip())

    for m in re.findall(r'"([^"]+)"', text):
        if m.strip():
            found.append(m.strip())

    uniq = []
    seen = set()
    for x in found:
        if x not in seen:
            uniq.append(x)
            seen.add(x)
    return uniq


def get_candidate_evidence_phrases(essay, criterion=None):
    essay = str(essay or "").strip()
    candidates = []

    # GRA evidence từ extractor hiện tại
    try:
        for c in extract_gra_error_candidates_v2(essay, max_candidates=8, max_per_type=2):
            snippet = str(c.get("snippet", "")).strip()
            if snippet:
                candidates.append(snippet)
    except Exception:
        pass

    # repeated word / awkward phrase
    for m in re.finditer(r"\b([A-Za-z]+)\s+\1\b", essay, flags=re.I):
        candidates.append(essay[m.start():m.end()])

    # một số phrase awkward phổ biến
    weak_patterns = [
        r"\bgive payment tax\b",
        r"\bpay tax to the state\b",
        r"\bthis veiw point\b",
        r"\bthe tax used in the most good jobs\b",
        r"\bwhile opthers disagree\b",
    ]
    low = essay.lower()
    for p in weak_patterns:
        m = re.search(p, low)
        if m:
            candidates.append(essay[m.start():m.end()])

    # giữ unique
    uniq = []
    seen = set()
    for x in candidates:
        x2 = re.sub(r"\s+", " ", str(x)).strip()
        if x2 and x2 not in seen:
            uniq.append(x2)
            seen.add(x2)

    return uniq[:10]


def is_minor_issue(issue):
    text = " ".join([
        str(issue.get("problem", "")),
        str(issue.get("reason", "")),
        str(issue.get("fix_instruction", "")),
    ]).lower()

    mild_patterns = [
        "transition",
        "logical flow",
        "smoother",
        "link the discussion",
        "paragraphing",
        "too generic",
        "more concrete example",
        "slightly generic",
        "needs one more anchored example",
    ]
    return any(p in text for p in mild_patterns)


def postprocess_verifier_issues(clean_issues, feedback):
    feedback = normalize_feedback_dict(feedback)

    gra_text = (
        feedback.get("GRA", {}).get("Limitation", "") + " " +
        feedback.get("GRA", {}).get("Revision", "")
    ).lower()

    out = []
    for issue in clean_issues:
        crit = issue.get("criterion", "")
        merged = " ".join([
            issue.get("problem", ""),
            issue.get("reason", ""),
            issue.get("fix_instruction", ""),
        ]).lower()

        if crit == "TR":
            overreach_patterns = [
                "social life",
                "relationships",
                "overall quality of life",
                "add new unrelated topic",
            ]
            if any(p in merged for p in overreach_patterns):
                continue

        if crit == "LR":
            # nếu verifier lấy cùng 1 phrase đã được xử lý ở GRA thì bỏ issue LR trùng
            gra_snips = extract_quoted_phrases(gra_text)
            if any(s.lower() in merged for s in gra_snips):
                continue

        out.append(issue)

    uniq = []
    seen = set()
    for x in out:
        key = (
            x.get("criterion", ""),
            x.get("problem", ""),
            x.get("reason", ""),
            x.get("fix_instruction", ""),
        )
        if key not in seen:
            seen.add(key)
            uniq.append(x)

    return uniq


def format_feedback_dict(feedback):
    if feedback is None:
        return "{}"
    try:
        return json.dumps(feedback, indent=2, ensure_ascii=False)
    except:
        return str(feedback)


def build_verify_feedback_prompt(prompt, essay, pred_scores, feedback):
    feedback_text = format_feedback_dict(feedback)

    return f"""
You are a strict IELTS Writing Task 2 feedback verifier.

Task:
Check whether the feedback is specific, evidence-based, criterion-faithful, and consistent with the FIXED predicted scores.
Do not rewrite the feedback. Only judge whether revision is needed.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Feedback to verify:
{feedback_text}

Verification rules:
1. Never change the scores.
2. Judge each criterion separately.
3. Mark revise only if there is a CLEAR and RELIABLE issue.
4. Prefer PASS unless the issue is clearly wrong, generic, off-scope, or internally inconsistent.
5. Do NOT over-penalize debatable points.

Consistency rules:
- Strength must not overstate quality compared with the fixed score.
- If the score is low, phrases such as `correct grammar`, `no major errors`, `precise language`, or `good range of vocabulary` may be too strong.
- If Limitation identifies several clear errors, Strength should remain conservative.
- Mark revise if Strength and Limitation clearly contradict each other.

Criterion-specific rules:

TR:
- Fail only if the feedback misunderstands the task or invents unrelated missing content.
- For agree/disagree prompts, do not require discussion of opposing views unless the task clearly asks for both views.

CC:
- Fail only if the feedback is fully generic or clearly off-scope.
- Do not fail merely for reasonable comments about transitions, flow, or paragraphing.
- If the essay already has multiple paragraphs, do not criticize it as if it had no paragraph structure.

LR:
- Fail only if the limitation is vague and names no concrete wording when discussing awkward or repetitive vocabulary.
- Mark revise if Strength overclaims lexical precision for a low score.

GRA:
- Fail if the limitation is generic and does not anchor itself to visible essay evidence.
- Mark revise if Strength claims grammar is correct or has no major errors while the limitation identifies several visible grammar problems.

Return JSON only:
{{
  "verdict": "pass" or "revise",
  "issues": [
    {{
      "criterion": "TR",
      "problem": "...",
      "reason": "...",
      "fix_instruction": "..."
    }}
  ]
}}
""".strip()


def normalize_verification_result(obj):
    if not isinstance(obj, dict):
        return {"verdict": "revise", "issues": []}

    verdict = str(obj.get("verdict", "revise")).strip().lower()
    if verdict not in ["pass", "revise"]:
        verdict = "revise"

    raw_issues = obj.get("issues", [])
    if not isinstance(raw_issues, list):
        raw_issues = []

    issues = []
    for x in raw_issues:
        if not isinstance(x, dict):
            continue
        crit = str(x.get("criterion", "")).upper().strip()
        if crit not in ["TR", "CC", "LR", "GRA"]:
            continue
        issues.append({
            "criterion": crit,
            "problem": str(x.get("problem", "")).strip(),
            "reason": str(x.get("reason", "")).strip(),
            "fix_instruction": str(x.get("fix_instruction", "")).strip(),
        })

    if verdict == "pass":
        issues = []

    return {
        "verdict": verdict,
        "issues": issues,
    }


def tool_verify_feedback(prompt, essay, state):
    pred_scores = state["pred_scores"]
    feedback = normalize_feedback_dict(state["feedback"])

    verify_prompt = build_verify_feedback_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        feedback=feedback,
    )

    raw = mistral_explain_writer(
        verify_prompt,
        temperature=0.0,
        max_new_tokens=700,
    )

    parsed = safe_json_loads(raw, default={})
    verification = normalize_verification_result(parsed)

    clean_issues = verification.get("issues", []) or []
    clean_issues = postprocess_verifier_issues(clean_issues, feedback)

    verdict = str(verification.get("verdict", "revise")).strip().lower()
    if verdict == "pass" and clean_issues:
        verdict = "revise"

    # mềm hơn cho demo: chỉ pass nếu issue thực sự rất nhẹ
    if verdict == "revise" and len(clean_issues) <= 2 and all(is_minor_issue(x) for x in clean_issues):
        verdict = "pass"
        clean_issues = []

    verification = {
        "verdict": verdict,
        "issues": clean_issues if verdict == "revise" else [],
        "raw": str(raw),
    }

    state["verification"] = verification

    return {
        "verdict": verification["verdict"],
        "num_issues": len(verification["issues"]),
        "issues": verification["issues"],
    }

Tool ReviseFeedBack

In [20]:
def build_revise_feedback_prompt(prompt, essay, pred_scores, feedback, verification):
    feedback_text = format_feedback_dict(feedback)
    issues_text = json.dumps(verification.get("issues", []), indent=2, ensure_ascii=False)

    return f"""
You are revising IELTS Writing Task 2 feedback after a verifier found problems.

Task:
Revise ONLY the criteria that appear in the issue list.
Keep all other criteria unchanged.

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Current feedback:
{feedback_text}

Verifier issues:
{issues_text}

Revision rules:
1. Revise ONLY the flagged criteria.
2. Keep unflagged criteria exactly unchanged.
3. Do not change any scores.
4. Do not mention any alternative band score.
5. Do not invent essay weaknesses not visible in the essay.
6. Maintain criterion boundaries:
   - TR only for task response/relevance/development
   - CC only for organization/cohesion
   - LR only for vocabulary
   - GRA only for grammar/sentence accuracy
7. For GRA, rewrite the Limitation so that it STARTS with concrete copied evidence from the essay.
8. If 2 or more visible GRA errors exist, mention at least 2 anchored examples.
9. Use backticks for copied evidence, for example:
   For example, `...` shows ... ; `...` shows ... .
10. IMPORTANT: the output must be valid JSON, so do NOT use raw double quotes inside JSON values.
11. Use backticks for copied essay evidence.
12. Preferred GRA pattern:
    For example, `...` shows ...
13. Output full JSON for all four criteria.
14. Output valid JSON only. No explanation before or after the JSON block.

Output valid JSON only in this format:
{{
  "TR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "CC": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "LR": {{
    "Strength": "...",
    "Limitation": "...",
    "Revision": "..."
  }},
  "GRA": {{
    "Strength": "...",
    "Limitation": "For example, `...` shows ...",
    "Revision": "..."
  }}
}}
""".strip()


def merge_feedback_keep_unflagged(old_feedback, new_feedback, flagged_criteria):
    merged = {}
    flagged_criteria = set(flagged_criteria)

    for crit in ["TR", "CC", "LR", "GRA"]:
        if crit in flagged_criteria:
            block = new_feedback.get(crit, {})
        else:
            block = old_feedback.get(crit, {})

        if not isinstance(block, dict):
            block = {}

        merged[crit] = {
            "Strength": str(block.get("Strength", "")).strip(),
            "Limitation": str(block.get("Limitation", "")).strip(),
            "Revision": str(block.get("Revision", "")).strip(),
        }

    return merged
def build_single_criterion_revision_prompt(
    criterion,
    prompt,
    essay,
    pred_scores,
    current_block,
    verifier_issue,
    evidence_phrases,
):
    evidence_text = "\n".join([f"- `{x}`" for x in evidence_phrases[:10]]) if evidence_phrases else "- None found"

    extra_rules = ""

    if criterion == "TR":
        extra_rules = """
TR revision rules:
- Do not invent new missing topics unless clearly required by the prompt.
- Prefer comments like `the idea is only briefly explained` or `the support is limited`.
- Do not ask for broader social discussion unless the task clearly demands it.
""".strip()

    elif criterion == "CC":
        extra_rules = """
CC revision rules:
- Focus only on organization, progression, paragraphing, and linking.
- Do not say `needs more transitions` unless you indicate where the flow weakens.
- Prefer specific comments such as abrupt shift, weak conclusion, or limited connection between paragraphs.
""".strip()

    elif criterion == "LR":
        extra_rules = """
LR revision rules:
- You MUST mention at least one concrete word or phrase from the essay in Limitation.
- Focus on repetition, imprecise wording, awkward phrasing, or collocation.
- Do not treat grammar-only mistakes as LR evidence.
""".strip()

    elif criterion == "GRA":
        extra_rules = """
GRA revision rules:
- You MUST mention at least one concrete visible phrase from the essay in Limitation.
- Explain the grammar problem directly.
- Revision should directly correct the phrase or sentence-level issue.
""".strip()

    return f"""
You are revising ONLY ONE IELTS Writing feedback criterion.

Criterion to revise: {criterion}

Fixed predicted scores:
TR={pred_scores["TR"]}, CC={pred_scores["CC"]}, LR={pred_scores["LR"]}, GRA={pred_scores["GRA"]}, Overall={pred_scores["Overall"]}

Essay prompt:
{prompt}

Essay:
{essay}

Current feedback block:
Strength: {current_block.get("Strength", "")}
Limitation: {current_block.get("Limitation", "")}
Revision: {current_block.get("Revision", "")}

Verifier issue:
Problem: {verifier_issue.get("problem", "")}
Reason: {verifier_issue.get("reason", "")}
Fix instruction: {verifier_issue.get("fix_instruction", "")}

Candidate visible evidence from the essay:
{evidence_text}

Strict rules:
1. Revise ONLY the criterion {criterion}.
2. Keep the score unchanged.
3. Return exactly 3 fields: Strength, Limitation, Revision.
4. Do not mention any other criterion.
5. Do not invent problems not visible in the essay.
6. Be specific, not generic.
7. Keep each field short and actionable.

{extra_rules}

Return JSON only:
{{
  "Strength": "...",
  "Limitation": "...",
  "Revision": "..."
}}
""".strip()

def revise_single_criterion(
    criterion,
    prompt,
    essay,
    pred_scores,
    feedback,
    verifier_issue,
):
    feedback = normalize_feedback_dict(feedback)

    current_block = feedback.get(criterion, {
        "Strength": "",
        "Limitation": "",
        "Revision": "",
    })

    evidence_phrases = get_candidate_evidence_phrases(essay, criterion=criterion)

    if criterion == "GRA" and not evidence_phrases:
        evidence_phrases = extract_quoted_phrases(current_block.get("Limitation", ""))[:3]

    revise_prompt = build_single_criterion_revision_prompt(
        criterion=criterion,
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        current_block=current_block,
        verifier_issue=verifier_issue,
        evidence_phrases=evidence_phrases,
    )

    raw = mistral_explain_writer(
        revise_prompt,
        temperature=0.0,
        max_new_tokens=350,
    )

    parsed = safe_json_loads(raw, default={})

    new_block = {
        "Strength": str(parsed.get("Strength", current_block.get("Strength", ""))).strip(),
        "Limitation": str(parsed.get("Limitation", current_block.get("Limitation", ""))).strip(),
        "Revision": str(parsed.get("Revision", current_block.get("Revision", ""))).strip(),
    }

    if criterion == "GRA":
        temp_feedback = {"GRA": new_block}
        temp_feedback = strengthen_gra_limitation(
            feedback=temp_feedback,
            essay=essay,
            min_anchors=2,
        )
        new_block = temp_feedback["GRA"]

    if not new_block["Strength"]:
        new_block["Strength"] = current_block.get("Strength", "")
    if not new_block["Limitation"]:
        new_block["Limitation"] = current_block.get("Limitation", "")
    if not new_block["Revision"]:
        new_block["Revision"] = current_block.get("Revision", "")

    feedback[criterion] = new_block
    return feedback

def tool_revise_feedback(prompt, essay, state):
    pred_scores = state["pred_scores"]
    feedback = normalize_feedback_dict(state["feedback"])
    verification = state["verification"]

    issues = verification.get("issues", []) if isinstance(verification, dict) else []
    if not issues:
        feedback = apply_feedback_consistency_rules(
            feedback=feedback,
            prompt=prompt,
            essay=essay,
            pred_scores=pred_scores,
        )
        state["feedback"] = feedback
        return {
            "revised": False,
            "flagged_criteria": [],
            "raw_preview": "",
        }

    flagged = []
    raw_logs = []

    for issue in issues:
        crit = str(issue.get("criterion", "")).upper().strip()
        if crit not in ["TR", "CC", "LR", "GRA"]:
            continue

        flagged.append(crit)
        before_block = dict(feedback.get(crit, {}))

        feedback = revise_single_criterion(
            criterion=crit,
            prompt=prompt,
            essay=essay,
            pred_scores=pred_scores,
            feedback=feedback,
            verifier_issue=issue,
        )

        after_block = feedback.get(crit, {})
        if after_block == before_block:
            after_block["Revision"] = issue.get(
                "fix_instruction",
                after_block.get("Revision", "")
            )
            feedback[crit] = after_block

        raw_logs.append({
            "criterion": crit,
            "issue": issue,
            "updated_block": feedback.get(crit, {}),
        })

    feedback = dedup_lr_gra_overlap(feedback)

    # NEW: deterministic consistency fix
    feedback = apply_feedback_consistency_rules(
        feedback=feedback,
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
    )
    state["feedback"] = feedback
    state["feedback_raw"] = json.dumps(raw_logs, ensure_ascii=False, indent=2)

    return {
        "revised": True,
        "flagged_criteria": flagged,
        "raw_preview": json.dumps(raw_logs, ensure_ascii=False)[:800],
    }

In [21]:
import json
import re

TOOL_DEFINITIONS = [
    {"name": "predict_scores", "description": "Predict IELTS criterion scores."},
    {"name": "detect_task_mismatch", "description": "Check whether the essay is off-topic or only partially addresses the prompt."},
    {"name": "check_argument_development", "description": "Check whether TR may be overestimated because the essay is too short, too general, or underdeveloped."},
    {"name": "check_coherence_risk", "description": "Check whether CC may be overestimated because paragraphing, progression, or linking are weak."},
    {"name": "check_lexical_risk", "description": "Check whether LR may be overestimated because vocabulary is repetitive or not varied enough."},
    {"name": "check_grammar_risk", "description": "Check whether GRA may be overestimated because there are too many visible grammar or sentence-control errors."},
    {"name": "retrieve_similar_essays", "description": "Retrieve similar essays from the vector database for grounding."},
    {"name": "generate_feedback", "description": "Generate criterion-level feedback grounded in the essay and retrieved cases."},
    {"name": "verify_feedback", "description": "Verify whether the feedback is faithful, specific, and criterion-correct."},
    {"name": "revise_feedback", "description": "Revise only problematic criteria after verification."},
    {"name": "check_high_band_consistency", "description": "Use retrieved high-band anchor essays to avoid over-penalizing genuinely strong essays after score guardrails."},
]


SCORE_CHECK_TOOL_MAP = {
    "TR": "check_argument_development",
    "CC": "check_coherence_risk",
    "LR": "check_lexical_risk",
    "GRA": "check_grammar_risk",
}


def summarize_agent_state(state):
    top_cases = state.get("top_cases", None)
    has_top_cases = False
    top_case_count = 0

    if top_cases is not None:
        try:
            top_case_count = len(top_cases)
            has_top_cases = top_case_count > 0
        except Exception:
            has_top_cases = False
            top_case_count = 0

    verification = state.get("verification") or {}
    verdict = verification.get("verdict", None)
    score_checks = state.get("score_checks", {}) or {}
    completed_checks = sorted([crit for crit, info in score_checks.items() if info])

    return {
        "has_pred_scores": state.get("pred_scores") is not None,
        "has_task_check": state.get("task_check") is not None,
        "has_top_cases": has_top_cases,
        "top_case_count": top_case_count,
        "score_checks_completed": completed_checks,
        "has_feedback": state.get("feedback") is not None,
        "has_verification": state.get("verification") is not None,
        "verification_verdict": verdict,
        "revision_count": state.get("revision_count", 0),
        "criterion_retry_count": state.get("criterion_retry_count", {}),
        "last_invalid_tool": state.get("last_invalid_tool", None),
    }


def _dedupe_keep_order(items):
    out = []
    seen = set()
    for x in items:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def _has_top_cases(state):
    top_cases = state.get("top_cases", None)
    if top_cases is None:
        return False
    try:
        return len(top_cases) > 0
    except Exception:
        return False


def _count_tool_calls(state, tool_name):
    trace = state.get("trace", []) or []
    return sum(1 for item in trace if str(item.get("chosen_tool", "")).strip() == tool_name)


def _verification_issue_text(state):
    verification = state.get("verification") or {}
    issues = verification.get("issues", []) or []
    chunks = []

    for issue in issues:
        if isinstance(issue, dict):
            for k in ["criterion", "reason", "problem", "message", "issue", "details"]:
                v = issue.get(k, "")
                if str(v).strip():
                    chunks.append(str(v))
        else:
            chunks.append(str(issue))

    raw = verification.get("raw", "")
    if str(raw).strip():
        chunks.append(str(raw))

    return " ".join(chunks).lower()


def _needs_more_grounding(state):
    """
    Cho phép retrieve thêm 1 lần ở pha revise nếu verifier chê:
    - thiếu evidence
    - chưa grounded
    - quá chung chung / not specific
    """
    text = _verification_issue_text(state)
    keywords = [
        "ground",
        "grounded",
        "evidence",
        "support",
        "specific",
        "specificity",
        "too general",
        "generic",
        "not concrete",
        "insufficient example",
        "insufficient examples",
        "not enough example",
        "not enough evidence",
    ]
    return any(k in text for k in keywords)


def _pending_score_check_tools(state):
    score_checks = state.get("score_checks", {}) or {}
    pending = []
    for crit in ["TR", "CC", "LR", "GRA"]:
        if not score_checks.get(crit):
            pending.append(SCORE_CHECK_TOOL_MAP[crit])
    return pending


def get_available_tools(state):
    """
    Phiên bản 'agentic nhưng an toàn':
    - Sau predict_scores: cho model chọn giữa detect_task_mismatch và retrieve_similar_essays
    - Task mismatch vẫn là bước bắt buộc trước khi generate_feedback
    - Trước khi generate feedback: chạy thêm các score guardrail tools cho TR/CC/LR/GRA
    - Nếu có tiêu chí band cao bị cap xuống sau guardrails, chạy check_high_band_consistency
      trước khi generate_feedback để tránh phạt oan bài mạnh
    - Khi verifier yêu cầu revise, có thể cho model chọn:
        + revise_feedback
        + retrieve_similar_essays (1 lần refresh) nếu issue liên quan grounding/evidence
    """
    has_pred_scores = state.get("pred_scores") is not None
    has_task_check = state.get("task_check") is not None
    has_top_cases = _has_top_cases(state)
    has_feedback = state.get("feedback") is not None
    has_verification = state.get("verification") is not None

    if not has_pred_scores:
        return ["predict_scores"]

    if not has_task_check:
        if not has_top_cases:
            return ["detect_task_mismatch", "retrieve_similar_essays"]
        return ["detect_task_mismatch"]

    pending_checks = _pending_score_check_tools(state)
    if pending_checks:
        if not has_top_cases:
            return _dedupe_keep_order(pending_checks + ["retrieve_similar_essays"])
        return pending_checks

    if not has_top_cases:
        return ["retrieve_similar_essays"]

    if not has_feedback:
        score_checks = state.get("score_checks", {}) or {}
        high_band_done = state.get("high_band_consistency") is not None
        need_high_band_check = False

        for crit in ["TR", "CC", "LR", "GRA"]:
            chk = score_checks.get(crit) or {}
            before = float(chk.get("score_before", 0.0) or 0.0)
            after = float(chk.get("score_after", before) or before)

            if before >= 7.0 and after < before:
                need_high_band_check = True
                break

        if need_high_band_check and not high_band_done:
            return ["check_high_band_consistency"]

        return ["generate_feedback"]

    if not has_verification:
        return ["verify_feedback"]

    verdict = str((state.get("verification") or {}).get("verdict", "")).strip().lower()

    if verdict == "revise":
        tools = []

        retry_map = state.get("criterion_retry_count", {}) or {}
        can_retry = any(v < 2 for v in retry_map.values()) if retry_map else state.get("revision_count", 0) < 2
        if can_retry:
            tools.append("revise_feedback")

        retrieve_calls = _count_tool_calls(state, "retrieve_similar_essays")
        if _needs_more_grounding(state) and retrieve_calls < 2:
            tools.append("retrieve_similar_essays")

        return _dedupe_keep_order(tools)

    if verdict == "pass":
        return []

    return ["verify_feedback"]


def fallback_policy(state):
    available = get_available_tools(state)
    if not available:
        return None

    priority = [
        "predict_scores",
        "detect_task_mismatch",
        "check_argument_development",
        "check_coherence_risk",
        "check_lexical_risk",
        "check_grammar_risk",
        "retrieve_similar_essays",
        "check_high_band_consistency",
        "generate_feedback",
        "verify_feedback",
        "revise_feedback",
    ]

    for p in priority:
        if p in available:
            return p
    return available[0]


def is_action_valid(action, state):
    return action in get_available_tools(state)


def build_agent_prompt(prompt, essay, state):
    available_tools = get_available_tools(state)
    state_summary = summarize_agent_state(state)

    invalid_note = ""
    if state.get("last_invalid_tool"):
        invalid_note = (
            f'Important: In the previous step, the tool "{state["last_invalid_tool"]}" was invalid.\n'
            f'Do NOT choose "{state["last_invalid_tool"]}" again unless it appears in Available tools.'
        )

    tool_desc = {x["name"]: x["description"] for x in TOOL_DEFINITIONS if x["name"] in available_tools}

    return f"""
You are an IELTS AES routing agent.

Your job is to choose the SINGLE best next tool for the current state.

Available tools:
{json.dumps(tool_desc, ensure_ascii=False, indent=2)}

Current state summary:
{json.dumps(state_summary, ensure_ascii=False, indent=2)}

{invalid_note}

Routing guidance:
- detect_task_mismatch is for checking prompt coverage / off-topic risk.
- check_argument_development is for checking whether TR looks too high for the level of idea development.
- check_coherence_risk is for checking whether CC looks too high because paragraphing, progression, or linking are weak.
- check_lexical_risk is for checking whether LR looks too high because the vocabulary is repetitive or limited.
- check_grammar_risk is for checking whether GRA looks too high because there are too many visible grammar problems.
- retrieve_similar_essays is for grounding feedback with similar essays.
- check_high_band_consistency is for using retrieved anchor essays to avoid penalizing genuinely strong essays too aggressively after guardrail checks.
- generate_feedback should only be chosen when enough context is ready.
- verify_feedback checks whether the current feedback is faithful and criterion-correct.
- revise_feedback should be chosen after verification fails, especially when the issues are already clear.
- If verification complains about weak evidence / weak grounding / lack of specificity, retrieving again can be a good choice.

Rules:
- Choose exactly ONE tool.
- The tool MUST be one of the Available tools.
- Prefer a tool that reduces uncertainty in the current state.
- Do not invent tool names.
- Output only these two lines:

Thought: <short reason>
Action: <tool_name>

Essay prompt:
{prompt}

Essay:
{essay}
""".strip()


def safe_parse_tool_call(raw_text):
    if raw_text is None:
        return None

    text = str(raw_text).strip()
    if not text:
        return None

    thought = ""
    tool_name = None

    m_thought = re.search(r"Thought\s*:\s*(.+)", text, flags=re.I)
    if m_thought:
        thought = m_thought.group(1).strip()

    m_action = re.search(r"Action\s*:\s*([A-Za-z_][A-Za-z0-9_]*)", text, flags=re.I)
    if m_action:
        tool_name = m_action.group(1).strip()

    if not tool_name:
        for t in [x["name"] for x in TOOL_DEFINITIONS]:
            if re.search(rf"\b{re.escape(t)}\b", text):
                tool_name = t
                break

    if not tool_name:
        return None

    return {
        "thought": thought,
        "tool_name": tool_name,
        "arguments": {}
    }


def choose_next_tool(prompt, essay, state, verbose=False):
    available_tools = get_available_tools(state)

    if not available_tools:
        return {
            "raw_decision": "[deterministic] no valid tool available",
            "parsed_tool_call": None,
            "chosen_tool": None,
            "arguments": {},
            "source": "none",
            "thought": "No valid tool is available.",
            "fallback_reason": "no_available_tool",
        }

    if len(available_tools) == 1:
        tool_name = available_tools[0]
        return {
            "raw_decision": f"[deterministic] only one valid tool: {tool_name}",
            "parsed_tool_call": {
                "thought": "Only one valid tool is available in the current state.",
                "tool_name": tool_name,
                "arguments": {},
            },
            "chosen_tool": tool_name,
            "arguments": {},
            "source": "rule",
            "thought": "Only one valid tool is available in the current state.",
            "fallback_reason": None,
        }

    agent_prompt = build_agent_prompt(prompt, essay, state)
    raw_decision = mistral_explain_writer(agent_prompt, temperature=0.0, max_new_tokens=120)
    parsed = safe_parse_tool_call(raw_decision)

    if parsed is None:
        return {
            "raw_decision": raw_decision,
            "parsed_tool_call": None,
            "chosen_tool": fallback_policy(state),
            "arguments": {},
            "source": "fallback",
            "thought": "",
            "fallback_reason": "parse_failed",
        }

    tool_name = parsed["tool_name"]

    if not is_action_valid(tool_name, state):
        return {
            "raw_decision": raw_decision,
            "parsed_tool_call": parsed,
            "chosen_tool": fallback_policy(state),
            "arguments": {},
            "source": "fallback",
            "thought": parsed.get("thought", ""),
            "fallback_reason": f"invalid_tool:{tool_name}",
        }

    return {
        "raw_decision": raw_decision,
        "parsed_tool_call": parsed,
        "chosen_tool": tool_name,
        "arguments": parsed.get("arguments", {}),
        "source": "model",
        "thought": parsed.get("thought", ""),
        "fallback_reason": None,
    }


def execute_tool_call(tool_name, arguments, prompt, essay, df, db_embeddings, state, top_k_vector=20, top_k_final=5):
    arguments = arguments or {}

    if tool_name == "predict_scores":
        return tool_predict_scores(prompt, essay, state)

    elif tool_name == "detect_task_mismatch":
        return tool_detect_task_mismatch(prompt, essay, state)

    elif tool_name == "check_argument_development":
        return tool_check_argument_development(prompt, essay, state)

    elif tool_name == "check_coherence_risk":
        return tool_check_coherence_risk(prompt, essay, state)

    elif tool_name == "check_lexical_risk":
        return tool_check_lexical_risk(prompt, essay, state)

    elif tool_name == "check_grammar_risk":
        return tool_check_grammar_risk(prompt, essay, state)

    elif tool_name == "retrieve_similar_essays":
        return tool_retrieve_similar_essays(
            prompt,
            essay,
            df,
            db_embeddings,
            state,
            top_k_vector=int(arguments.get("top_k_vector", top_k_vector)),
            top_k_final=int(arguments.get("top_k_final", top_k_final)),
        )
    elif tool_name == "check_high_band_consistency":
        return tool_check_high_band_consistency(prompt, essay, state)

    elif tool_name == "generate_feedback":
        return tool_generate_feedback(prompt, essay, state)

    elif tool_name == "verify_feedback":
        return tool_verify_feedback(prompt, essay, state)

    elif tool_name == "revise_feedback":
        issues = state.get("verification", {}).get("issues", [])
        for issue in issues:
            crit = str(issue.get("criterion", "")).upper().strip()
            if crit in state.get("criterion_retry_count", {}):
                state["criterion_retry_count"][crit] += 1

        result = tool_revise_feedback(prompt, essay, state)
        state["verification"] = None
        return result

    else:
        raise ValueError(f"Unknown tool: {tool_name}")



In [22]:
CRITERIA = ["TR", "CC", "LR", "GRA"]
DEMO_FORCE_PASS_AFTER_REVISE = False

def run_agent_feedback_pipeline(
    prompt,
    essay,
    df,
    db_embeddings,
    max_steps=12,
    top_k_vector=20,
    top_k_final=5,
    verbose=True,
):
    state = {
        "pred_scores": None,
        "task_check": None,
        "score_checks": {},
        "top_cases": None,
        "feedback": None,
        "verification": None,
        "trace": [],
        "revision_count": 0,
        "criterion_retry_count": {c: 0 for c in CRITERIA},
        "done": False,
        "done_reason": None,
        "last_invalid_tool": None,
    }

    for step in range(max_steps):
        decision = choose_next_tool(prompt, essay, state, verbose=verbose)

        tool_name = decision["chosen_tool"]
        arguments = decision["arguments"]
        source = decision["source"]
        thought = decision["thought"]
        fallback_reason = decision["fallback_reason"]
        raw_decision = decision["raw_decision"]
        parsed = decision["parsed_tool_call"]

        trace_item = {
            "step": step + 1,
            "raw_decision": raw_decision,
            "parsed_tool_call": parsed,
            "chosen_tool": tool_name,
            "arguments": arguments,
            "source": source,
            "thought": thought,
            "fallback_reason": fallback_reason,
        }

        if fallback_reason and str(fallback_reason).startswith("invalid_tool:"):
            state["last_invalid_tool"] = str(fallback_reason).split(":", 1)[1]
        else:
            state["last_invalid_tool"] = None

        if verbose:
            print("\n" + "=" * 90)
            print(f"[Step {step+1}] DECISION")
            print("chosen_tool   :", tool_name)
            print("source        :", source)
            print("thought       :", thought)
            print("fallback      :", fallback_reason)
            print("raw_decision  :", raw_decision)

        if tool_name is None:
            trace_item["tool_result"] = {"status": "No valid action available."}
            state["trace"].append(trace_item)
            state["done_reason"] = "No valid action available."
            break

        try:
            tool_result = execute_tool_call(
                tool_name=tool_name,
                arguments=arguments,
                prompt=prompt,
                essay=essay,
                df=df,
                db_embeddings=db_embeddings,
                state=state,
                top_k_vector=top_k_vector,
                top_k_final=top_k_final,
            )
            trace_item["tool_result"] = tool_result

            if verbose:
                print(f"[Step {step+1}] TOOL RESULT")
                try:
                    print(json.dumps(tool_result, indent=2, ensure_ascii=False))
                except Exception:
                    print(tool_result)

        except Exception as e:
            trace_item["tool_result"] = {
                "error": str(e),
                "traceback": traceback.format_exc(limit=2),
            }
            state["trace"].append(trace_item)
            state["done_reason"] = f"Tool execution failed at step {step+1}: {tool_name}"
            break

        state["trace"].append(trace_item)

        if tool_name == "verify_feedback" and state.get("verification") is not None:
            verdict = str(state["verification"].get("verdict", "")).strip().lower()

            if (
                DEMO_FORCE_PASS_AFTER_REVISE
                and verdict == "revise"
                and state.get("revision_count", 0) >= 1
            ):
                state["verification"] = {
                    "verdict": "pass",
                    "issues": [],
                    "raw": str(state["verification"]),
                    "note": "demo override after at least one revise cycle"
                }
                verdict = "pass"

            if verdict == "pass":
                state["done"] = True
                state["done_reason"] = "Feedback verified successfully."
                break

        if tool_name == "revise_feedback":
            state["revision_count"] += 1

        if state["revision_count"] >= 2:
            state["done"] = True
            state["done_reason"] = "Stopped after max revisions."
            break

    if not state["done"] and not state["done_reason"]:
        state["done_reason"] = "Stopped by max_steps."

    return state



In [23]:
import os
import glob

def find_retrieval_csv():
    candidate_files = [
        "/content/ielts_train_df2.csv",
        "/content/drive/MyDrive/ielts_train_df2.csv",
        "/content/drive/MyDrive/ielts_train_aug_df.csv",
        "/content/ielts_train_aug_df.csv",
    ]
    for f in candidate_files:
        if os.path.exists(f):
            return f

    for root in ["/content", "/content/drive/MyDrive"]:
        if not os.path.exists(root):
            continue
        hits = glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
        hits = [h for h in hits if "train" in os.path.basename(h).lower() and "ielts" in os.path.basename(h).lower()]
        if hits:
            return sorted(hits)[0]

    raise FileNotFoundError("Không tìm thấy file train CSV để build retrieval database.")

TRAIN_CSV = find_retrieval_csv()
print("Using TRAIN_CSV =", TRAIN_CSV)
df_retrieval, db_embeddings = build_retrieval_database(TRAIN_CSV)

quality_index = build_quality_index(df_retrieval, embedding_model, store_name="main_quality")
print("Quality-aware retrieval index ready.")


Using TRAIN_CSV = /content/drive/MyDrive/ielts_train_aug_df.csv


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

[quality-index] TR: 7777 cases


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

[quality-index] CC: 7777 cases


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

[quality-index] LR: 7777 cases


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

[quality-index] GRA: 7777 cases
Quality-aware retrieval index ready.


In [24]:
# =========================
# NOTEBOOK SANITY CHECK
# =========================

REQUIRED_SYMBOLS = [
    "tool_predict_scores",
    "tool_detect_task_mismatch",
    "tool_check_argument_development",
    "tool_check_coherence_risk",
    "tool_check_lexical_risk",
    "tool_check_grammar_risk",
    "tool_retrieve_similar_essays",
    "tool_generate_feedback",
    "tool_verify_feedback",
    "tool_revise_feedback",
    "execute_tool_call",
    "choose_next_tool",
    "run_agent_feedback_pipeline",
    "ensure_retrieval_columns",
    "extract_gra_error_candidates_v2",
    "tool_check_high_band_consistency",
]
missing_symbols = [name for name in REQUIRED_SYMBOLS if name not in globals()]
if missing_symbols:
    raise NameError(f"Notebook is missing required symbols: {missing_symbols}")

print("Sanity check passed. All required tool and agent functions are defined.")


Sanity check passed. All required tool and agent functions are defined.


In [ ]:
# =========================
# FULL REPLACEMENT DEMO CELL (PRETTIER UI)
# =========================

import html
import json
import pandas as pd
import gradio as gr


# ---------- helpers ----------
def escape_html(x):
    return html.escape("" if x is None else str(x))


def make_json_safe(obj):
    try:
        return json.loads(json.dumps(obj, ensure_ascii=False, default=str))
    except Exception:
        return {"error": "state is not JSON serializable", "raw": str(obj)}


def feedback_is_empty(feedback):
    if not isinstance(feedback, dict):
        return True

    for crit in ["TR", "CC", "LR", "GRA"]:
        block = feedback.get(crit, {})
        if not isinstance(block, dict):
            continue
        if any(str(block.get(k, "")).strip() for k in ["Strength", "Limitation", "Revision"]):
            return False
    return True


def find_latest_feedback_raw_from_trace(state):
    trace = state.get("trace", []) or []
    for item in reversed(trace):
        tool = str(item.get("chosen_tool", "")).strip()
        tool_result = item.get("tool_result", {}) or {}
        if tool in ["revise_feedback", "generate_feedback"]:
            raw_preview = tool_result.get("raw_preview", "")
            if str(raw_preview).strip():
                return str(raw_preview)
    return ""


def get_status_class(verdict):
    verdict = str(verdict or "").strip().lower()
    if verdict in ["pass", "done", "success"]:
        return "ok"
    if verdict in ["partial_mismatch", "revise", "warning"]:
        return "warn"
    if verdict in ["off_topic", "error", "failed"]:
        return "bad"
    return "muted"


def render_badge(text, status="muted"):
    return f'<span class="badge {escape_html(status)}">{escape_html(text)}</span>'


def render_empty_card(title, subtitle):
    return f"""
    <div class="ui-card">
        <div class="section-title">{escape_html(title)}</div>
        <div class="muted">{escape_html(subtitle)}</div>
    </div>
    """


def render_summary_html(state):
    trace = state.get("trace", []) or []
    pred_scores = state.get("pred_scores", {}) or {}
    done = state.get("done", False)
    done_reason = state.get("done_reason", "") or "N/A"
    revision_count = state.get("revision_count", 0)
    verification = state.get("verification", {}) or {}
    verification_verdict = verification.get("verdict", "")
    task_check = state.get("task_check", {}) or {}
    task_verdict = task_check.get("verdict", "")

    done_badge = render_badge("Done" if done else "Running/Stopped", "ok" if done else "warn")
    verify_badge = render_badge(
        verification_verdict if verification_verdict else "N/A",
        get_status_class(verification_verdict)
    )
    task_badge = render_badge(
        task_verdict if task_verdict else "N/A",
        get_status_class(task_verdict)
    )

    overall = pred_scores.get("Overall", "—")

    return f"""
    <div class="ui-card hero-card">
        <div class="hero-top">
            <div>
                <div class="eyebrow">IELTS AES Agent</div>
                <div class="hero-title">Execution Summary</div>
                <div class="hero-subtitle">{escape_html(done_reason)}</div>
            </div>
            <div class="hero-overall">
                <div class="hero-overall-label">Overall</div>
                <div class="hero-overall-score">{escape_html(overall)}</div>
            </div>
        </div>

        <div class="stats-grid">
            <div class="stat-tile">
                <div class="stat-label">Steps</div>
                <div class="stat-value">{len(trace)}</div>
            </div>
            <div class="stat-tile">
                <div class="stat-label">Revisions</div>
                <div class="stat-value">{revision_count}</div>
            </div>
            <div class="stat-tile">
                <div class="stat-label">Finished</div>
                <div class="stat-value small">{done_badge}</div>
            </div>
            <div class="stat-tile">
                <div class="stat-label">Verification</div>
                <div class="stat-value small">{verify_badge}</div>
            </div>
        </div>

        <div class="chip-row">
            <div class="chip-block"><span class="chip-label">Task check</span>{task_badge}</div>
            <div class="chip-block"><span class="chip-label">Verification</span>{verify_badge}</div>
        </div>
    </div>
    """


def render_scores_html(pred_scores):
    if not pred_scores:
        return render_empty_card("Scores", "Chưa có điểm.")

    def score_card(label, value, accent="blue"):
        return f"""
        <div class="score-card {accent}">
            <div class="score-name">{escape_html(label)}</div>
            <div class="score-value">{escape_html(value)}</div>
        </div>
        """

    return f"""
    <div class="ui-card">
        <div class="section-title">Predicted Scores</div>
        <div class="scores-grid">
            {score_card("TR", pred_scores.get("TR", "—"), "blue")}
            {score_card("CC", pred_scores.get("CC", "—"), "violet")}
            {score_card("LR", pred_scores.get("LR", "—"), "teal")}
            {score_card("GRA", pred_scores.get("GRA", "—"), "orange")}
            {score_card("Overall", pred_scores.get("Overall", "—"), "pink")}
        </div>
    </div>
    """


def render_task_check_html(task_check, pred_scores=None):
    if not task_check:
        return render_empty_card("Task Check", "Chưa có kết quả task check.")

    verdict = str(task_check.get("verdict", "")).strip()
    reason = str(task_check.get("reason", "")).strip()
    evidence = str(task_check.get("evidence", "")).strip()
    missing_parts = task_check.get("missing_parts", []) or []

    badge = render_badge(verdict or "N/A", get_status_class(verdict))

    missing_html = ""
    if missing_parts:
        items = "".join([f"<li>{escape_html(x)}</li>" for x in missing_parts])
        missing_html = f"""
        <div class="subsection">
            <div class="sub-label">Missing parts</div>
            <ul class="nice-list">{items}</ul>
        </div>
        """
    else:
        missing_html = """
        <div class="subsection">
            <div class="sub-label">Missing parts</div>
            <div class="muted">None</div>
        </div>
        """

    tr_html = ""
    if pred_scores:
        tr_html = f"""
        <div class="mini-grid">
            <div class="mini-card">
                <div class="mini-label">TR after guardrail</div>
                <div class="mini-value">{escape_html(pred_scores.get("TR", "N/A"))}</div>
            </div>
            <div class="mini-card">
                <div class="mini-label">Overall</div>
                <div class="mini-value">{escape_html(pred_scores.get("Overall", "N/A"))}</div>
            </div>
        </div>
        """

    return f"""
    <div class="ui-card">
        <div class="section-head">
            <div class="section-title">Task Check</div>
            {badge}
        </div>

        <div class="subsection">
            <div class="sub-label">Reason</div>
            <div class="body-text">{escape_html(reason or "N/A")}</div>
        </div>

        <div class="subsection">
            <div class="sub-label">Evidence</div>
            <div class="body-text">{escape_html(evidence or "N/A")}</div>
        </div>

        {missing_html}
        {tr_html}
    </div>
    """


def render_feedback_html(feedback, feedback_raw=""):
    if not feedback or feedback_is_empty(feedback):
        if str(feedback_raw).strip():
            return f"""
            <div class="ui-card">
                <div class="section-title">Feedback</div>
                <div class="warn-box">Structured parse failed. Đây là raw output từ model.</div>
                <pre class="pretty-pre">{escape_html(feedback_raw)}</pre>
            </div>
            """
        return render_empty_card("Feedback", "Chưa có feedback.")

    crit_colors = {
        "TR": "blue",
        "CC": "violet",
        "LR": "teal",
        "GRA": "orange",
    }

    blocks = []
    for crit in ["TR", "CC", "LR", "GRA"]:
        block = feedback.get(crit, {}) or {}
        strength = str(block.get("Strength", "")).strip() or "—"
        limitation = str(block.get("Limitation", "")).strip() or "—"
        revision = str(block.get("Revision", "")).strip() or "—"

        blocks.append(
            f"""
            <div class="feedback-card {crit_colors.get(crit, 'blue')}">
                <div class="feedback-head">
                    <div class="feedback-crit">{escape_html(crit)}</div>
                </div>

                <div class="feedback-item">
                    <div class="feedback-label">Strength</div>
                    <div class="feedback-text">{escape_html(strength)}</div>
                </div>

                <div class="feedback-item">
                    <div class="feedback-label">Limitation</div>
                    <div class="feedback-text">{escape_html(limitation)}</div>
                </div>

                <div class="feedback-item">
                    <div class="feedback-label">Revision</div>
                    <div class="feedback-text">{escape_html(revision)}</div>
                </div>
            </div>
            """
        )

    return f"""
    <div class="ui-card">
        <div class="section-title">Criterion Feedback</div>
        <div class="feedback-grid">
            {''.join(blocks)}
        </div>
    </div>
    """


def render_verification_html(verification):
    if not verification:
        return render_empty_card("Verification", "Chưa có verification.")

    verdict = str(verification.get("verdict", "")).strip()
    issues = verification.get("issues", []) or []
    badge = render_badge(verdict or "N/A", get_status_class(verdict))

    if not issues:
        issues_html = '<div class="success-box">No issues. Feedback passed verification.</div>'
    else:
        cards = []
        for i, issue in enumerate(issues, start=1):
            criterion = str(issue.get("criterion", "")).strip()
            problem = str(issue.get("problem", "")).strip()
            reason = str(issue.get("reason", "")).strip()
            fix_instruction = str(issue.get("fix_instruction", "")).strip()

            cards.append(
                f"""
                <div class="issue-card">
                    <div class="issue-top">
                        <div class="issue-index">#{i}</div>
                        <div class="issue-criterion">{escape_html(criterion)}</div>
                    </div>
                    <div class="issue-row"><span>Problem</span>{escape_html(problem or "N/A")}</div>
                    <div class="issue-row"><span>Reason</span>{escape_html(reason or "N/A")}</div>
                    <div class="issue-row"><span>Fix</span>{escape_html(fix_instruction or "N/A")}</div>
                </div>
                """
            )
        issues_html = f'<div class="issues-grid">{"".join(cards)}</div>'

    return f"""
    <div class="ui-card">
        <div class="section-head">
            <div class="section-title">Verification</div>
            {badge}
        </div>
        {issues_html}
    </div>
    """


def prepare_cases_df(top_cases):
    if top_cases is None:
        return pd.DataFrame()

    try:
        df = pd.DataFrame(top_cases)
    except Exception:
        return pd.DataFrame()

    if len(df) == 0:
        return pd.DataFrame()

    out = df.copy()

    for c in ["prompt", "essay", "evaluation", "doc_text"]:
        if c in out.columns:
            out[c] = out[c].astype(str).str.replace("\n", " ", regex=False).str.slice(0, 240)

    preferred = ["TR", "CC", "LR", "GRA", "Overall", "final_score", "prompt", "essay"]
    cols = [c for c in preferred if c in out.columns]

    if not cols:
        return out.reset_index(drop=True)

    return out[cols].reset_index(drop=True)


def render_cases_info_html(top_cases):
    try:
        n = len(top_cases or [])
    except Exception:
        n = 0

    if n == 0:
        return render_empty_card("Retrieved Cases", "Không có retrieved cases.")

    return f"""
    <div class="ui-card">
        <div class="section-head">
            <div class="section-title">Retrieved Cases</div>
            {render_badge(f"{n} cases", "ok")}
        </div>
        <div class="body-text">
            Các bài tương tự đã được truy xuất từ vector database để grounding feedback.
        </div>
    </div>
    """


def render_trace_html(trace):
    if not trace:
        return """
        <div class="trace-wrap">
            <div class="trace-card">
                <div class="trace-title">No trace</div>
            </div>
        </div>
        """

    blocks = []
    for t in trace:
        step = t.get("step", "")
        tool = t.get("chosen_tool", "")
        source = t.get("source", "")
        thought = t.get("thought", "")
        fallback_reason = t.get("fallback_reason", "")
        raw_decision = t.get("raw_decision", "")
        parsed_tool_call = t.get("parsed_tool_call", {})
        tool_result = t.get("tool_result", {})

        try:
            parsed_pretty = json.dumps(parsed_tool_call, indent=2, ensure_ascii=False, default=str)
        except Exception:
            parsed_pretty = str(parsed_tool_call)

        try:
            tool_result_pretty = json.dumps(tool_result, indent=2, ensure_ascii=False, default=str)
        except Exception:
            tool_result_pretty = str(tool_result)

        tool_badge = render_badge(tool or "N/A", "ok")
        source_badge = render_badge(source or "N/A", "muted")

        blocks.append(
            f"""
            <div class="trace-card">
                <div class="trace-head">
                    <div class="trace-title">Step {escape_html(step)}</div>
                    <div class="trace-badges">{tool_badge}{source_badge}</div>
                </div>

                <div class="trace-line"><span>Thought</span>{escape_html(thought or "N/A")}</div>
                <div class="trace-line"><span>Fallback</span>{escape_html(fallback_reason or "None")}</div>

                <details>
                    <summary>Raw decision</summary>
                    <pre class="trace-pre">{escape_html(raw_decision)}</pre>
                </details>

                <details>
                    <summary>Parsed tool call</summary>
                    <pre class="trace-pre">{escape_html(parsed_pretty)}</pre>
                </details>

                <details open>
                    <summary>Tool result</summary>
                    <pre class="trace-pre">{escape_html(tool_result_pretty)}</pre>
                </details>
            </div>
            """
        )

    return f'<div class="trace-wrap">{"".join(blocks)}</div>'


def run_demo_pipeline(prompt, essay):
    if not str(prompt).strip() or not str(essay).strip():
        empty_html = render_empty_card("Thiếu dữ liệu", "Vui lòng nhập cả đề bài và essay.")
        empty_df = pd.DataFrame()
        empty_trace = """
        <div class="trace-wrap">
            <div class="trace-card">
                <div class="trace-title">No trace</div>
            </div>
        </div>
        """
        return (
            empty_html,   # summary_out
            empty_html,   # feedback_preview_out
            empty_html,   # scores_out
            empty_html,   # task_out
            empty_html,   # feedback_tab_out
            empty_html,   # verification_out
            empty_df,     # cases_df_out
            empty_html,   # cases_info_out
            empty_trace,  # trace_out
            {},           # state_out
        )

    state = run_agent_feedback_pipeline(
        prompt=prompt,
        essay=essay,
        df=df_retrieval,
        db_embeddings=db_embeddings,
        max_steps=10,
        top_k_vector=20,
        top_k_final=5,
        verbose=False,
    )
    feedback_raw = state.get("feedback_raw", "")
    if not str(feedback_raw).strip():
        feedback_raw = find_latest_feedback_raw_from_trace(state)

    summary_html = render_summary_html(state)
    scores_html = render_scores_html(state.get("pred_scores"))
    task_html = render_task_check_html(state.get("task_check"), state.get("pred_scores"))
    feedback_html = render_feedback_html(state.get("feedback"), feedback_raw)
    verification_html = render_verification_html(state.get("verification"))
    cases_df = prepare_cases_df(state.get("top_cases"))
    cases_info_html = render_cases_info_html(state.get("top_cases"))
    trace_html = render_trace_html(state.get("trace", []))
    safe_state = make_json_safe(state)

    return (
        summary_html,       # summary_out
        feedback_html,      # feedback_preview_out
        scores_html,        # scores_out
        task_html,          # task_out
        feedback_html,      # feedback_tab_out
        verification_html,  # verification_out
        cases_df,           # cases_df_out
        cases_info_html,    # cases_info_out
        trace_html,         # trace_out
        safe_state,         # state_out
    )


# ---------- CSS ----------
DEMO_CSS = """
:root{
  --bg: #08111f;
  --bg-2: #0b1426;
  --panel: rgba(14, 24, 43, 0.88);
  --panel-2: rgba(17, 28, 49, 0.96);
  --panel-soft: rgba(20, 33, 57, 0.85);
  --border: rgba(110, 140, 190, 0.18);
  --text: #e8eefc;
  --muted: #95a3c7;
  --blue: #60a5fa;
  --violet: #a78bfa;
  --teal: #2dd4bf;
  --orange: #f59e0b;
  --pink: #f472b6;
  --red: #f87171;
  --green: #34d399;
  --yellow: #fbbf24;
  --shadow: 0 10px 30px rgba(0,0,0,.28);
}

.gradio-container{
  background:
    radial-gradient(circle at top left, rgba(37,99,235,.20), transparent 28%),
    radial-gradient(circle at top right, rgba(168,85,247,.14), transparent 24%),
    linear-gradient(180deg, #07101d 0%, #0a1220 100%) !important;
  color: var(--text) !important;
}

.gradio-container, .gradio-container *{
  color: var(--text);
}

.block, .gr-box, .gr-panel, .gr-form{
  background: transparent !important;
  border-color: transparent !important;
  box-shadow: none !important;
}

label, .gradio-container .prose, .gr-markdown, .gr-form *{
  color: var(--text) !important;
}

textarea, input{
  background: rgba(11, 19, 35, 0.92) !important;
  color: var(--text) !important;
  border: 1px solid rgba(120,145,190,.18) !important;
  border-radius: 16px !important;
  box-shadow: inset 0 1px 0 rgba(255,255,255,.02) !important;
}

textarea:focus, input:focus{
  border-color: rgba(96,165,250,.55) !important;
  box-shadow: 0 0 0 3px rgba(96,165,250,.12) !important;
}

button.primary, button[variant="primary"]{
  background: linear-gradient(90deg, #2563eb 0%, #4f46e5 100%) !important;
  color: white !important;
  border: none !important;
  border-radius: 14px !important;
  min-height: 46px !important;
  box-shadow: 0 10px 24px rgba(37,99,235,.28) !important;
}

button.primary:hover, button[variant="primary"]:hover{
  filter: brightness(1.05);
}

.ui-card{
  background: linear-gradient(180deg, rgba(17,28,49,.96) 0%, rgba(13,22,39,.94) 100%);
  border: 1px solid var(--border);
  border-radius: 22px;
  padding: 18px;
  box-shadow: var(--shadow);
}

.hero-card{
  padding: 22px;
}

.hero-top{
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  gap: 20px;
  margin-bottom: 18px;
}

.eyebrow{
  font-size: 12px;
  letter-spacing: .12em;
  text-transform: uppercase;
  color: #8fb7ff;
  margin-bottom: 6px;
}

.hero-title{
  font-size: 28px;
  font-weight: 800;
  line-height: 1.15;
  margin-bottom: 6px;
}

.hero-subtitle{
  color: var(--muted);
  font-size: 14px;
}

.hero-overall{
  min-width: 120px;
  text-align: center;
  background: linear-gradient(180deg, rgba(96,165,250,.15), rgba(168,85,247,.12));
  border: 1px solid rgba(120,145,190,.22);
  border-radius: 20px;
  padding: 14px 16px;
}

.hero-overall-label{
  color: var(--muted);
  font-size: 12px;
  margin-bottom: 6px;
}

.hero-overall-score{
  font-size: 34px;
  font-weight: 800;
}

.section-head{
  display:flex;
  align-items:center;
  justify-content:space-between;
  gap:12px;
  margin-bottom: 14px;
}

.section-title{
  font-size: 18px;
  font-weight: 800;
  margin-bottom: 12px;
}

.stats-grid{
  display:grid;
  grid-template-columns: repeat(4, minmax(0,1fr));
  gap: 12px;
}

.stat-tile{
  background: rgba(255,255,255,.025);
  border: 1px solid rgba(120,145,190,.15);
  border-radius: 16px;
  padding: 14px;
}

.stat-label{
  color: var(--muted);
  font-size: 12px;
  margin-bottom: 8px;
}

.stat-value{
  font-size: 22px;
  font-weight: 800;
}

.stat-value.small{
  font-size: 15px;
  font-weight: 600;
}

.chip-row{
  display:flex;
  gap:12px;
  flex-wrap:wrap;
  margin-top: 14px;
}

.chip-block{
  display:flex;
  align-items:center;
  gap:10px;
  background: rgba(255,255,255,.025);
  border: 1px solid rgba(120,145,190,.14);
  padding: 10px 12px;
  border-radius: 14px;
}

.chip-label{
  color: var(--muted);
  font-size: 13px;
}

.badge{
  display:inline-flex;
  align-items:center;
  gap:8px;
  padding: 6px 10px;
  border-radius: 999px;
  font-size: 12px;
  font-weight: 700;
  border: 1px solid transparent;
  white-space: nowrap;
}

.badge.ok{
  background: rgba(52,211,153,.10);
  color: #9ff2cf;
  border-color: rgba(52,211,153,.22);
}

.badge.warn{
  background: rgba(251,191,36,.10);
  color: #ffe08a;
  border-color: rgba(251,191,36,.22);
}

.badge.bad{
  background: rgba(248,113,113,.11);
  color: #ffb1b1;
  border-color: rgba(248,113,113,.22);
}

.badge.muted{
  background: rgba(148,163,184,.10);
  color: #d3dceb;
  border-color: rgba(148,163,184,.16);
}

.scores-grid{
  display:grid;
  grid-template-columns: repeat(5, minmax(0,1fr));
  gap: 12px;
}

.score-card{
  border-radius: 18px;
  padding: 16px;
  border: 1px solid rgba(255,255,255,.06);
  background: rgba(255,255,255,.025);
}

.score-card.blue{ box-shadow: inset 0 0 0 1px rgba(96,165,250,.08); }
.score-card.violet{ box-shadow: inset 0 0 0 1px rgba(167,139,250,.08); }
.score-card.teal{ box-shadow: inset 0 0 0 1px rgba(45,212,191,.08); }
.score-card.orange{ box-shadow: inset 0 0 0 1px rgba(245,158,11,.08); }
.score-card.pink{ box-shadow: inset 0 0 0 1px rgba(244,114,182,.08); }

.score-name{
  color: var(--muted);
  font-size: 13px;
  margin-bottom: 10px;
}

.score-value{
  font-size: 28px;
  font-weight: 800;
  line-height: 1;
}

.subsection{
  margin-top: 12px;
  padding: 14px;
  border-radius: 16px;
  background: rgba(255,255,255,.025);
  border: 1px solid rgba(120,145,190,.12);
}

.sub-label{
  font-size: 12px;
  text-transform: uppercase;
  letter-spacing: .06em;
  color: #9bb2e2;
  margin-bottom: 8px;
}

.body-text{
  line-height: 1.65;
  color: var(--text);
}

.muted{
  color: var(--muted);
}

.nice-list{
  margin: 0;
  padding-left: 18px;
  color: var(--text);
}

.mini-grid{
  display:grid;
  grid-template-columns: repeat(2, minmax(0,1fr));
  gap: 12px;
  margin-top: 12px;
}

.mini-card{
  background: rgba(255,255,255,.025);
  border: 1px solid rgba(120,145,190,.12);
  padding: 14px;
  border-radius: 14px;
}

.mini-label{
  color: var(--muted);
  font-size: 12px;
  margin-bottom: 8px;
}

.mini-value{
  font-size: 22px;
  font-weight: 800;
}

.feedback-grid{
  display:grid;
  grid-template-columns: repeat(2, minmax(0,1fr));
  gap: 14px;
}

.feedback-card{
  border-radius: 18px;
  padding: 16px;
  border: 1px solid rgba(120,145,190,.14);
  background: rgba(255,255,255,.025);
}

.feedback-card.blue{ box-shadow: inset 0 0 0 1px rgba(96,165,250,.08); }
.feedback-card.violet{ box-shadow: inset 0 0 0 1px rgba(167,139,250,.08); }
.feedback-card.teal{ box-shadow: inset 0 0 0 1px rgba(45,212,191,.08); }
.feedback-card.orange{ box-shadow: inset 0 0 0 1px rgba(245,158,11,.08); }

.feedback-head{
  display:flex;
  align-items:center;
  justify-content:space-between;
  margin-bottom: 10px;
}

.feedback-crit{
  font-size: 18px;
  font-weight: 800;
}

.feedback-item{
  padding: 12px 0;
  border-top: 1px solid rgba(255,255,255,.06);
}

.feedback-item:first-of-type{
  border-top: 0;
}

.feedback-label{
  color: #9bb2e2;
  font-size: 12px;
  text-transform: uppercase;
  letter-spacing: .06em;
  margin-bottom: 7px;
}

.feedback-text{
  line-height: 1.65;
  color: var(--text);
  white-space: pre-wrap;
}

.pretty-pre{
  background: #08101d;
  border: 1px solid rgba(120,145,190,.16);
  color: #dbe7ff;
  padding: 14px;
  border-radius: 16px;
  white-space: pre-wrap;
  word-break: break-word;
  overflow-x: auto;
}

.warn-box{
  background: rgba(251,191,36,.10);
  color: #ffe08a;
  border: 1px solid rgba(251,191,36,.18);
  padding: 12px 14px;
  border-radius: 14px;
  margin-bottom: 12px;
}

.success-box{
  background: rgba(52,211,153,.10);
  color: #b3f5d8;
  border: 1px solid rgba(52,211,153,.18);
  padding: 14px;
  border-radius: 14px;
}

.issues-grid{
  display:grid;
  grid-template-columns: repeat(2, minmax(0,1fr));
  gap: 12px;
}

.issue-card{
  background: rgba(255,255,255,.025);
  border: 1px solid rgba(120,145,190,.14);
  border-radius: 16px;
  padding: 14px;
}

.issue-top{
  display:flex;
  align-items:center;
  gap:10px;
  margin-bottom: 12px;
}

.issue-index{
  width: 28px;
  height: 28px;
  border-radius: 999px;
  display:flex;
  align-items:center;
  justify-content:center;
  background: rgba(248,113,113,.14);
  color: #ffb1b1;
  font-weight: 800;
  font-size: 12px;
}

.issue-criterion{
  font-weight: 800;
  font-size: 16px;
}

.issue-row{
  display:grid;
  grid-template-columns: 72px 1fr;
  gap: 10px;
  margin-top: 8px;
  line-height: 1.55;
}

.issue-row span{
  color: #9bb2e2;
  font-weight: 700;
  font-size: 12px;
  text-transform: uppercase;
}

.trace-wrap{
  display:flex;
  flex-direction:column;
  gap:12px;
}

.trace-card{
  border: 1px solid rgba(120,145,190,.16);
  border-radius: 18px;
  padding: 16px;
  background: linear-gradient(180deg, rgba(17,28,49,.96) 0%, rgba(11,20,37,.94) 100%);
  box-shadow: var(--shadow);
}

.trace-head{
  display:flex;
  align-items:center;
  justify-content:space-between;
  gap:12px;
  margin-bottom: 10px;
}

.trace-badges{
  display:flex;
  gap:8px;
  flex-wrap:wrap;
}

.trace-title{
  font-weight: 800;
  font-size: 18px;
}

.trace-line{
  display:grid;
  grid-template-columns: 84px 1fr;
  gap: 10px;
  margin-top: 8px;
  line-height: 1.5;
}

.trace-line span{
  color: #9bb2e2;
  font-weight: 700;
  font-size: 12px;
  text-transform: uppercase;
}

.trace-card details{
  margin-top: 12px;
  border-top: 1px solid rgba(255,255,255,.06);
  padding-top: 12px;
}

.trace-card summary{
  cursor: pointer;
  font-weight: 700;
  color: #a8c5ff;
}

.trace-pre{
  white-space: pre-wrap;
  word-break: break-word;
  background: #07101d;
  color: #dbe7ff;
  padding: 12px;
  border-radius: 14px;
  border: 1px solid rgba(120,145,190,.16);
  margin-top: 10px;
  font-size: 13px;
  line-height: 1.55;
  overflow-x: auto;
}

.gradio-container .tab-nav{
  background: rgba(255,255,255,.02) !important;
  border: 1px solid rgba(120,145,190,.12) !important;
  border-radius: 16px !important;
  padding: 6px !important;
}

.gradio-container .tab-nav button{
  border-radius: 12px !important;
}

.gradio-container .tab-nav button.selected{
  background: linear-gradient(90deg, rgba(37,99,235,.22), rgba(79,70,229,.18)) !important;
}

footer{
  display:none !important;
}

@media (max-width: 1100px){
  .scores-grid{
    grid-template-columns: repeat(2, minmax(0,1fr));
  }
  .feedback-grid, .issues-grid{
    grid-template-columns: 1fr;
  }
  .stats-grid{
    grid-template-columns: repeat(2, minmax(0,1fr));
  }
}

@media (max-width: 700px){
  .hero-top{
    flex-direction: column;
  }
  .mini-grid{
    grid-template-columns: 1fr;
  }
}
"""


# ---------- close old demo if exists ----------
if 'demo' in globals():
    try:
        globals()['demo'].close()
    except Exception:
        pass


# ---------- UI ----------
with gr.Blocks(
    title="IELTS AES Agent Demo",
    theme=gr.themes.Soft(),
    css=DEMO_CSS
) as demo:
    gr.HTML("""
    <div class="ui-card hero-card" style="margin-bottom:14px;">
        <div class="eyebrow">Demo Interface</div>
        <div class="hero-title">IELTS Writing Task 2 AES Agent</div>
        <div class="hero-subtitle">
            Scoring · Task Guardrail · Retrieval · Feedback · Verification · Revision
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            prompt_in = gr.Textbox(
                label="Essay Prompt",
                lines=7,
                placeholder="Paste IELTS Writing Task 2 prompt here..."
            )
            essay_in = gr.Textbox(
                label="Essay",
                lines=18,
                placeholder="Paste essay here..."
            )
            run_btn = gr.Button("Run Demo", variant="primary")

        with gr.Column(scale=5):
            summary_out = gr.HTML()
            feedback_preview_out = gr.HTML()

    with gr.Tabs():
        with gr.Tab("Scores & Task Check"):
            scores_out = gr.HTML()
            task_out = gr.HTML()

        with gr.Tab("Feedback"):
            feedback_tab_out = gr.HTML()
            verification_out = gr.HTML()

        with gr.Tab("Retrieved Cases"):
            cases_info_out = gr.HTML()
            cases_df_out = gr.Dataframe(
                label="Top retrieved cases",
                interactive=False
            )

        with gr.Tab("Trace"):
            trace_out = gr.HTML()

        with gr.Tab("Raw State"):
            state_out = gr.JSON(label="Agent state")

    run_btn.click(
        fn=run_demo_pipeline,
        inputs=[prompt_in, essay_in],
        outputs=[
            summary_out,
            feedback_preview_out,
            scores_out,
            task_out,
            feedback_tab_out,
            verification_out,
            cases_df_out,
            cases_info_out,
            trace_out,
            state_out,
        ],
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_47320/3103388973.py:1087: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_47320/3103388973.py:1087: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abf416f6d25da5c0c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o